<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu Jupyter Notebook'un Türkçesini Colab'da açınız](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-02/NB02_makine_ogrenmesi_temelleri.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-02/ders.html)

# Week 02: Machine learning fundamentals

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lab.html) · [PDF notes](Week02_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/NB02_machine_learning_fundamentals.ipynb)

</div>

**How to use this Jupyter notebook.** The notebook follows the practice stops of the lecture page. Every stop has three parts, and each part also runs on its own: Python basics, application with ready prompts, and the ready application. **Python basics** teaches the Python that the stop needs, with small examples from the topic of the week; its exercises check themselves. **Prompt and application** gives ready prompts for an artificial intelligence (AI) assistant: The assistant writes the code, you run it, and a check cell tells you whether the result is right. **Ready application** is the complete working Python program of the stop, with notes that explain it.

Go through the three parts in this order. If you already know Python, you may start with the prompts; if you only want to see the result, open the ready application. The weekly project at the end stands on its own as well.

**Start cells.** Every part opens with a start cell. It prepares the course package and the packages of the week, and it repeats the data calls and the code of the earlier parts that the cells of the part use. The part therefore runs without errors even when you start there. If you have run the earlier parts in this session, you do not need to run the start cell; running it again does no harm. A few sections of the ready applications, which the lecture page opens directly, have a start cell of their own.

**Data.** Every cell that reads a data set offers three sources: `"library"` reads the data with a Python library from the site of the data set, `"web"` reads the copy in the GitHub repository of the course, and `"computer"` reads a file that you have downloaded. In Google Colab, choose the source in the box on the right of the cell; in Jupyter, change the line `DATA_SOURCE = ...`. With `"computer"`, Jupyter looks for the file in the folder of this notebook, and Colab opens a window to upload it.

**Finding your place.** The table of contents lists every stop and part. In Colab it opens with the list icon on the left; in Jupyter, with View → Table of Contents. Every part ends with a link back to the lecture page. In Colab, your changes are kept only in a copy saved to Drive: Use File → Save a copy in Drive.

**Where it runs.** The notebook runs in Jupyter (Anaconda) on your own computer and in Google Colab. Every data set here is synthetic or comes from a public research database. Real patient data belong only on a computer of your institution, under its rules, and never in a cloud notebook or in an AI chat.

## 0. Setup

**First use of Jupyter Notebook.** A notebook is made of cells. This cell holds text; the grey cells hold Python code. Click into the code cell below and press Shift+Enter: The cell runs, its result appears below it, and the number in the brackets on its left, such as `[1]`, shows the order of the run. The kernel, the Python process behind the notebook, remembers everything that the cells have run, so the cells run from top to bottom. **Kernel → Restart Kernel and Run All Cells** repeats the whole notebook, and Ctrl+S (Cmd+S on a Mac) saves it as an `.ipynb` file.

**The start cell.** The code cell below prepares the session. It finds the course package `vclinic` in the course folder; if the notebook was opened on its own, for example in Google Colab, it downloads the package from GitHub once. It then loads the packages, chooses the language of the messages and prepares the expected answers of the exercises. Its last lines import the libraries of the week. Every part of the notebook opens with a start cell that holds the same setup, together with the data calls of the part.

In [ ]:
# Start cell of the Jupyter notebook
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the libraries of the week

from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, mean_absolute_error,
                             precision_recall_curve, roc_auc_score, roc_curve,
                             silhouette_score)
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from vclinic.datasets import load_cleveland

## Stop 1: Mini project: A CAD prediction model, ROC curve and AUC

*Lecture page: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#stop-1)*

The Cleveland patients of the UCI Heart Disease data set had an exercise test and a coronary angiography. In this stop a logistic regression is built from eight variables of the history and the exercise test, and its ability to recognise coronary artery disease is checked in patients that the model has not seen.

**At the end of this stop:** A model of coronary artery disease with a test AUC of 0.931, its sensitivity at the threshold 0.5 (0.8) and the odds ratio of the highest heart rate in the exercise test (0.58).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 1 · Python basics

A model of scikit-learn needs two things: a table of numbers with one row per patient, called `X`, and the outcome of each patient, called `y`. This part turns the Cleveland data of Week 1 into such a table. It selects columns and rows, handles missing values, turns a category into numbers, splits the patients into a training set and a test set, and fits a first model. Every later section of the notebook uses these steps.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 1 · Python basics», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week02 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland

In [ ]:
# Python practice: a small example for learning the language
heart = load_cleveland()
print(heart.shape)
heart[["age", "chol", "disease"]].head()

**Python note.** `heart.shape` gives the number of rows and columns. Double brackets, `heart[["age", "chol"]]`, select several columns and return a smaller DataFrame. Single brackets, `heart["age"]`, select one column and return a *Series*, a single column of values with the same row labels.

In [ ]:
# Python practice: a small example for learning the language
older = heart[heart["age"] >= 60]
print(len(older), "patients aged 60 or more")
heart.loc[heart["sex"] == 0, ["age", "chol", "disease"]].head(3)

**Python note.** `heart["age"] >= 60` compares every age with 60 and gives a column of `True` and `False`, a *Boolean mask*. Putting the mask in brackets keeps the rows where it is `True`. `.loc[rows, columns]` selects rows and columns at the same time: here the women (`sex == 0`) and three columns.

In [ ]:
# Python practice: a small example for learning the language
print(heart.isna().sum()[heart.isna().sum() > 0])
filled = heart["ca"].fillna(heart["ca"].median())
print("Missing in ca after filling:", filled.isna().sum())
print("Rows without any missing value:", len(heart.dropna()))

**Python note.** A missing value appears as `NaN` (not a number). `isna()` marks it, and `.sum()` counts the marks of each column. `fillna(value)` replaces missing values, here with the median of the column, and `dropna()` removes every row with a missing value. The variables of this week's model have no missing values, so the notebook keeps all 303 patients.

In [ ]:
# Python practice: a small example for learning the language
pd.get_dummies(heart["cp"], prefix="cp", drop_first=True, dtype=int).head()

**Python note.** The column `cp` codes the chest pain: 1 typical angina, 2 atypical angina, 3 non-anginal pain and 4 no pain. The numbers are names, not amounts: Pain of type 4 is not twice pain of type 2. `pd.get_dummies` therefore makes one *dummy variable* (0 or 1) per type. `drop_first=True` leaves out type 1, which becomes the reference: A patient with zeros in all three columns has typical angina.

In [ ]:
# The table of the model: one row per patient, numbers only
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
data = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"],
                      drop_first=True, dtype=int)
X = data.drop(columns="disease")
y = data["disease"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
print("X:", X.shape, " training set:", X_train.shape, " test set:", X_test.shape)
print("Share with disease: training", round(y_train.mean(), 3),
      " test", round(y_test.mean(), 3))

**Python note.** `train_test_split` returns four objects, and the line `X_train, X_test, y_train, y_test = ...` unpacks them into four names in this order. `test_size=0.25` keeps a quarter of the patients aside, `stratify=y` keeps the share of patients with disease the same in both parts, and `random_state` makes the split the same at every run. Here each patient has one row. When a patient has several rows, for example several ECGs, the split must keep all rows of a patient on the same side, which `GroupShuffleSplit` does with the patient identifier.

In [ ]:
# Python practice: a small example for learning the language
first = LogisticRegression(max_iter=1000)
first.fit(X_train, y_train)
print(first.predict(X_test[:5]))
print(first.predict_proba(X_test[:5]).round(3))

**Python note.** Every model of scikit-learn follows the same pattern. `fit(X, y)` learns from the training data. `predict(X)` returns a label, 0 or 1, and `predict_proba(X)` returns one column of probabilities per class: The first column is the probability of 0, the second of 1. `[:5]` takes the first five rows, and `max_iter=1000` lets the fitting run long enough.

In [ ]:
# Python practice: a small example for learning the language
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
p_test = model.predict_proba(X_test)[:, 1]
print("Probabilities of the first five test patients:", p_test[:5].round(3))

**Python note.** `make_pipeline` chains steps into one model. `StandardScaler` turns each column into a *z-score*, $(x - \bar{x})/s$, with the mean and the standard deviation of the training data. Inside the pipeline, `fit` learns these values from the training patients only, and the test patients are scaled with the same numbers, so nothing of the test set leaks into the model. `[:, 1]` takes all rows (`:`) and the second column (`1`) of a two-dimensional array.

**Quick check.** A model returns `predict_proba(X_test)` with shape `(76, 2)`. What do the 76 and the 2 mean?

<details><summary>Answer</summary>

76 rows are the 76 patients of the test set. The 2 columns are the probabilities of class 0 and class 1, which add up to 1 in every row.

</details>

**Exercise 1.1.** How many cells of `heart` are missing in total, over all rows and columns?

In [ ]:
data = exercise_table("1.1", heart=heart)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 1.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(heart.isna().sum().sum())
teach.check("Exercise 1.1", answer)

**Exercise 1.2.** How many patients go to the test set when the 303 patients are split with `test_size=0.25`? Answer with `len(...)`.

In [ ]:
data = exercise_table("1.2", heart=heart)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 1.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = len(X_test)
teach.check("Exercise 1.2", answer)

**Exercise 1.3.** What is the accuracy of `model`, the share of correctly labelled patients, on the test set? Use `model.score(X_test, y_test)` and round to three decimals.

In [ ]:
data = exercise_table("1.3", X=X, X_test=X_test, y=y)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 1.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(model.score(X_test, y_test), 3)
teach.check("Exercise 1.3", answer)

**«Stop 1 · Python basics» is complete.** Next comes «Stop 1 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#decision-trees-and-random-forests) and continue reading at «Comparing a decision tree and a random forest».

### Stop 1 · Prompt and application

This part has three prompts: «The model and its test AUC», «Sensitivity at the threshold 0.5» and «The odds ratio of the highest heart rate».

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Working with prompts.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab and opens with the Gemini button at the top of the window; ChatGPT, Claude, Copilot or Gemini in another browser tab work as well. All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md), where GitHub shows a copy button next to each one.

**Two names in every prompt: "data" and "answer".** An AI assistant usually does not see the notebook, so every prompt describes the data in words. For each prompt, the data cell puts its data into a table named "data", and the prompt describes the rows and the columns of this table. The prompt also asks the assistant to keep the result under the name "answer", and the check cell reads the result under this name. These are the only two names from the code that appear in the prompts of the course.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

**Start cell.** For «Stop 1 · Prompt and application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland

DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
heart = load_cleveland(source=DATA_SOURCE)
_FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
_coded = pd.get_dummies(heart[_FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
_X, _y = _coded.drop(columns="disease"), _coded["disease"]
_Xtr, _Xte, _ytr, _yte = train_test_split(_X, _y, test_size=0.25, stratify=_y, random_state=RANDOM_STATE)
_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(_Xtr, _ytr)
_CP = {1: "typical angina", 2: "atypical angina", 3: "non-anginal pain", 4: "asymptomatic"}


def _cad_table():
    """The Cleveland patients with the eight variables of the model and the outcome."""
    return pd.DataFrame({"age": heart["age"].astype(int), "sex": heart["sex"].astype(int),
                         "chest pain": heart["cp"].map(_CP), "resting blood pressure": heart["trestbps"].astype(int),
                         "cholesterol": heart["chol"].astype(int), "highest heart rate": heart["thalach"].astype(int),
                         "exercise angina": heart["exang"].astype(int), "ST depression": heart["oldpeak"],
                         "CAD": heart["disease"].astype(int)})

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, mean_absolute_error,
                             precision_recall_curve, roc_auc_score, roc_curve,
                             silhouette_score)

#### Prompt 1.1: The model and its test AUC

**Goal.** Fit a logistic regression on the training set and measure it on the test set with the area under the ROC curve (AUC).

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Among 303 patients who had an exercise test and a coronary angiography at the Cleveland Clinic, the cardiologist wants a model that predicts coronary artery disease from eight variables of the history and the exercise test, tested on patients that the model has not seen.  
> Data: The notebook holds a table named "data". Each row is a patient: "age" gives the age in years, and "sex" is 1 for men and 0 for women. The column "chest pain" says "typical angina", "atypical angina", "non-anginal pain" or "asymptomatic". "resting blood pressure" gives the resting systolic blood pressure in mm Hg, "cholesterol" the serum cholesterol in mg/dL and "highest heart rate" the highest heart rate of the exercise test in beats per minute. "exercise angina" is 1 when angina occurred during exercise and 0 otherwise, and "ST depression" gives the ST depression during exercise compared with rest in mm. "CAD" is 1 when at least one major coronary artery was narrowed by more than 50 percent of its diameter at angiography, and 0 otherwise. The column "set" says "training" or "test"; the test set holds 76 patients.  
> Task: Turn the type of chest pain into three yes/no columns, with typical angina as the reference. Use the eight variables other than "CAD" and "set" as the inputs of the model. Standardise them with the means and the standard deviations of the training patients only. Fit a logistic regression on the training set and compute the area under the ROC curve (AUC) of the probabilities that it gives to the test patients.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the AUC of the test set with three decimals and keep it under the name "answer".  
> Test: The test set must hold 76 patients, and the AUC must lie between 0.5 and 1.  

In [ ]:
# Data of Prompt 1.1: the table "data" that the prompt describes
data = _cad_table()
data["set"] = np.where(data.index.isin(_Xte.index), "test", "training")
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.1
teach.check_step("Prompt 1.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.1 (run it only if the code of the assistant does not pass the check)
data = _cad_table()
data["set"] = np.where(data.index.isin(_Xte.index), "test", "training")
X_table = data[["age", "sex", "resting blood pressure", "cholesterol",
                    "highest heart rate", "exercise angina", "ST depression"]].copy()
for pain in ["atypical angina", "non-anginal pain", "asymptomatic"]:
    X_table[pain] = (data["chest pain"] == pain).astype(int)
outcome = data["CAD"]
test_rows = data["set"] == "test"
lr_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr_model.fit(X_table[~test_rows], outcome[~test_rows])
probabilities = lr_model.predict_proba(X_table[test_rows])[:, 1]
print("Patients in the test set:", int(test_rows.sum()))
answer = round(roc_auc_score(outcome[test_rows], probabilities), 3)
print("Test AUC:", answer)
teach.check_step("Prompt 1.1", "answer", globals())

**What the result shows.** The model ranks a randomly chosen patient with coronary artery disease above a randomly chosen patient without it in about 93 of 100 such pairs. A test set of 76 patients still leaves this estimate uncertain; Stop 2 repeats the test in another way.

#### Prompt 1.2: Sensitivity at the threshold 0.5

**Goal.** Turn the probabilities of the model into decisions at the threshold 0.5 and find out how many of the patients with disease the model recognises.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know which share of the patients with coronary artery disease a model recognises when it calls a patient positive at a probability of 0.5 or more.  
> Data: The notebook holds a table named "data". Each row is a patient of the test set: The column "probability" gives the probability of coronary artery disease that the model gives to the patient, and "CAD" the true state, 1 for disease and 0 for none.  
> Task: Call a patient positive when the probability is 0.5 or more. Count the true positives (TP), the patients with disease who are called positive, and the false negatives (FN), the patients with disease who are called negative. The sensitivity is TP / (TP + FN).  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print TP, FN and the sensitivity with three decimals, and keep the sensitivity under the name "answer".  
> Test: TP + FN must equal the number of patients with disease in the table, which is 35.  

In [ ]:
# Data of Prompt 1.2: the table "data" that the prompt describes
data = pd.DataFrame({"probability": _model.predict_proba(_Xte)[:, 1].round(3), "CAD": _yte.to_numpy()})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.2
teach.check_step("Prompt 1.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.2 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"probability": _model.predict_proba(_Xte)[:, 1].round(3), "CAD": _yte.to_numpy()})
positive = data["probability"] >= 0.5
diseased = data["CAD"] == 1
tp = int((positive & diseased).sum())
fn = int((~positive & diseased).sum())
answer = round(tp / (tp + fn), 3)
print("TP =", tp, " FN =", fn, " sensitivity =", answer)
teach.check_step("Prompt 1.2", "answer", globals())

**What the result shows.** At this threshold the model misses about one in five patients with disease. A lower threshold finds more of them at the price of more false alarms; Part B of the interactive lab shows this trade-off.

#### Prompt 1.3: The odds ratio of the highest heart rate

**Goal.** Read from the model how the odds of coronary artery disease change when the highest heart rate of the exercise test is one standard deviation higher.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see, as odds ratios, how the variables of a model of coronary artery disease go with the odds of disease, above all the highest heart rate of the exercise test.  
> Data: The notebook holds a table named "data". Each row is a variable of a logistic regression that was fitted on standardised variables: The column "variable" gives its name, and "coefficient" the coefficient of the model, the change in the log odds for an increase of one standard deviation. The coefficients of the types of chest pain compare each type with typical angina.  
> Task: Turn every coefficient into an odds ratio per standard deviation: e raised to the power of the coefficient. Print all odds ratios with the names of their variables.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the odds ratio of the highest heart rate with two decimals and keep it under the name "answer".  
> Test: A coefficient of 0 must give an odds ratio of exactly 1, and the odds ratio of the ST depression must be above 1.  

In [ ]:
# Data of Prompt 1.3: the table "data" that the prompt describes
_NAMES = {"age": "age", "sex": "sex (male)", "trestbps": "resting blood pressure",
          "chol": "cholesterol", "thalach": "highest heart rate", "exang": "exercise angina",
          "oldpeak": "ST depression", "cp_2.0": "chest pain: atypical angina",
          "cp_3.0": "chest pain: non-anginal pain", "cp_4.0": "chest pain: asymptomatic"}
data = pd.DataFrame({"variable": [_NAMES[c] for c in _X.columns],
                     "coefficient": _model.named_steps["logisticregression"].coef_[0].round(4)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.3
teach.check_step("Prompt 1.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.3 (run it only if the code of the assistant does not pass the check)
_NAMES = {"age": "age", "sex": "sex (male)", "trestbps": "resting blood pressure",
          "chol": "cholesterol", "thalach": "highest heart rate", "exang": "exercise angina",
          "oldpeak": "ST depression", "cp_2.0": "chest pain: atypical angina",
          "cp_3.0": "chest pain: non-anginal pain", "cp_4.0": "chest pain: asymptomatic"}
data = pd.DataFrame({"variable": [_NAMES[c] for c in _X.columns],
                     "coefficient": _model.named_steps["logisticregression"].coef_[0].round(4)})
data["odds ratio"] = np.exp(data["coefficient"]).round(2)
print(data)
answer = float(data.loc[data["variable"] == "highest heart rate", "odds ratio"].iloc[0])
print("Odds ratio of the highest heart rate per SD:", answer)
teach.check_step("Prompt 1.3", "answer", globals())

**What the result shows.** The odds ratio is below 1: A higher heart rate in the exercise test goes with lower odds of coronary artery disease, as a good chronotropic response suggests clinically. The ratio refers to one standard deviation of the highest heart rate in the training patients, not to one beat per minute.

**«Stop 1 · Prompt and application» is complete.** Next comes «Stop 1 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#decision-trees-and-random-forests) and continue reading at «Comparing a decision tree and a random forest».

### Stop 1 · Ready application

This part holds the complete mini project of the session: The Cleveland data are loaded from one of three sources, the table of the model is built and the pipeline of standardisation and logistic regression learns from the training patients. A regression line, the odds ratios of the model and the probability of the case of the week follow, then the confusion matrix, the ROC curve and the four measures at three thresholds.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 1 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week02 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, mean_absolute_error,
                             precision_recall_curve, roc_auc_score, roc_curve,
                             silhouette_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

#### A first model: Logistic regression for coronary artery disease

*Lecture page: [Logistic regression: Predicting the risk of CAD](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#logistic-regression-from-risk-factors-to-a-probability)*

The mini project of the session builds a model that estimates the probability of coronary artery disease (CAD) from seven variables and the type of chest pain of the 303 Cleveland patients. The model is fitted on the training set and studied here. The next part, «Measuring a model», measures it on the test set.

**From the formulas to the code.** The linear predictor $z = \beta_0 + \sum_j \beta_j x_j$ uses `intercept_` for $\beta_0$ and `coef_` for the $\beta_j$. The probability $p = 1/(1 + e^{-z})$ is what `predict_proba` returns in its second column. The odds ratio of a variable is $e^{\beta_j}$, `np.exp(coef_)`. After `StandardScaler`, the $x_j$ are z-scores, so each odds ratio refers to one standard deviation (SD) of its variable.

**The data and the model of the mini project.** The cells below load the Cleveland data from one of three sources, build the table of the model and fit the pipeline of standardisation and logistic regression on the training patients, as «Stop 1 · Python basics» does step by step. The rest of the application works with this model.

In [ ]:
# Loading the Cleveland data from one of three sources
DATA_SOURCE = "web"   #@param ["library", "web", "computer"]

COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
FILE = "processed.cleveland.data"
URL = vclinic.COURSE_RAW + "data/uci-heart-disease/" + FILE   # the copy of the UCI file in the course repository

if DATA_SOURCE == "library":
    # ucimlrepo, the Python library of the UCI Machine Learning Repository, downloads the data set by its number
    vclinic.install("ucimlrepo")
    from ucimlrepo import fetch_ucirepo
    uci = fetch_ucirepo(id=45)
    heart = pd.concat([uci.data.features, uci.data.targets], axis=1)[COLUMNS]
elif DATA_SOURCE == "web":
    # pandas reads the file straight from its web address
    heart = pd.read_csv(URL, header=None, names=COLUMNS, na_values="?")
else:
    # the file on this computer: download it first with the link in the note below this cell
    path = vclinic.local_file(FILE)   # Jupyter looks in the folder of this notebook; Colab opens an upload window
    heart = pd.read_csv(path, header=None, names=COLUMNS, na_values="?")

heart["disease"] = (heart["num"] > 0).astype(int)   # 1 when a major vessel is narrowed by more than 50 percent
print("source:", DATA_SOURCE, "| rows and columns:", heart.shape)
heart.head()

**Python note.** The same table can come from three sources. `fetch_ucirepo(id=45)` asks the UCI repository for its data set number 45 and returns the variables and the outcome as two tables, which `pd.concat` puts side by side. `pd.read_csv` reads a file from a web address just as it reads one from the computer: `header=None` says that the file has no row of column names, `names=COLUMNS` gives the names, and `na_values="?"` turns the question marks of the file into missing values.

**Reading the data from your own computer.** For the source `"computer"`, first download [uci_heart_disease_cleveland.zip](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/raw/main/data/downloads/uci_heart_disease_cleveland.zip). Open the zip file and put the file `processed.cleveland.data` in the same folder as this Jupyter notebook. Then choose `"computer"` as the source and run the data cell again. In Google Colab, the cell shows a button for choosing the file on your computer. The zip file also holds the citation and the license of the data.

In [ ]:
# The table of the model: one row per patient, numbers only
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
data = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"],
                      drop_first=True, dtype=int)
X = data.drop(columns="disease")
y = data["disease"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
print("X:", X.shape, " training set:", X_train.shape, " test set:", X_test.shape)
print("Share with disease: training", round(y_train.mean(), 3),
      " test", round(y_test.mean(), 3))

In [ ]:
# The model: standardisation and logistic regression in one pipeline
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
p_test = model.predict_proba(X_test)[:, 1]
print("Probabilities of the first five test patients:", p_test[:5].round(3))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-app1-model)*

**Before the classification: a regression line.** *Lecture page: [The modelling pipeline with scikit-learn](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#the-modelling-pipeline-with-scikit-learn)*

A regression model predicts a number. The cell below fits the line $\hat{y} = b_0 + b_1 x$ of the maximum heart rate in the exercise test on age with least squares, as in the worked example of the lecture, and compares it with the rule of 220 minus age. The next cell builds a pipeline of three steps for a regression with four variables, one of which has missing values.

**On the lecture page:** [Worked example: Maximum heart rate and age in the Cleveland data](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#worked-example-maximum-heart-rate-and-age-in-the-cleveland-data). The cell below fits the same line to the 303 patients.

In [ ]:
# A regression line: the maximum heart rate in the exercise test from age
reg = LinearRegression().fit(heart[["age"]], heart["thalach"])
b0, b1 = reg.intercept_, reg.coef_[0]
print(f"b0 = {b0:.2f} bpm, b1 = {b1:.3f} bpm per year, R² = {reg.score(heart[['age']], heart['thalach']):.3f}")
print("Prediction for a patient aged 60:",
      round(float(reg.predict(pd.DataFrame({"age": [60]}))[0]), 1), "bpm")
ages = np.array([29, 77])
fig, ax = plt.subplots(figsize=(5.8, 4.2))
ax.scatter(heart["age"], heart["thalach"], s=10, color="#888888")
ax.plot(ages, b0 + b1 * ages, color="#9B1B1B", label="least squares")
ax.plot(ages, 220 - ages, "--", color="#1F5F8B", label="220 minus age")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Maximum heart rate (bpm)")
ax.legend()
plt.show()

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-c116)*

**Python note.** `heart[["age"]]`, with double brackets, is a table with one column, because a model of scikit-learn always expects a table of variables. `reg.intercept_` is $b_0$ and `reg.coef_[0]` is $b_1$. `reg.score` returns $R^2$ for a regression model. `reg.predict` needs a table with the same column name, so the new patient is written as `pd.DataFrame({"age": [60]})`.

In [ ]:
# A pipeline with three steps: filling missing values, standardisation and the model
REG_COLS = ["age", "trestbps", "chol", "ca"]          # ca has 4 missing values
h_train, h_test = train_test_split(heart, test_size=0.25, random_state=RANDOM_STATE)
reg_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LinearRegression())
reg_pipe.fit(h_train[REG_COLS], h_train["thalach"])
print(reg_pipe)
print("Median of ca learned from the training patients:",
      reg_pipe.named_steps["simpleimputer"].statistics_[3])
mae = mean_absolute_error(h_test["thalach"], reg_pipe.predict(h_test[REG_COLS]))
print("Mean absolute error on the test set:", round(mae, 1), "bpm")

**Python note.** `make_pipeline` chains the three steps. `SimpleImputer(strategy="median")` learns the median of each column from the training patients and fills the missing values with it, even in the test patients. `print(reg_pipe)` lists the steps, and `named_steps["simpleimputer"].statistics_` holds the medians that the first step learned. `mean_absolute_error` is the mean of the absolute differences between the true and the predicted values.

In [ ]:
# Coefficients of the model as odds ratios per standard deviation
lr = model.named_steps["logisticregression"]
coefs = pd.DataFrame({"beta": lr.coef_[0], "odds ratio": np.exp(lr.coef_[0])},
                     index=X.columns).sort_values("beta")
print("Intercept beta_0 =", round(lr.intercept_[0], 3))
coefs.round(3)

**Python note.** `model.named_steps` holds the steps of a pipeline by name, so `named_steps["logisticregression"]` is the fitted logistic regression. `coef_[0]` is its first and only row of coefficients. `sort_values("beta")` orders the table from the most protective to the most harmful variable.

In [ ]:
# The case of the week on the treadmill: z and p by hand and by the model
case = pd.DataFrame([{"age": 57, "sex": 1, "trestbps": 119, "chol": 230, "thalach": 132,
                      "exang": 1, "oldpeak": 1.6, "cp_2.0": 1, "cp_3.0": 0, "cp_4.0": 0}])[X.columns]
scaler = model.named_steps["standardscaler"]
zscores = (case.to_numpy()[0] - scaler.mean_) / scaler.scale_
z = lr.intercept_[0] + np.sum(lr.coef_[0] * zscores)
p_hand = 1 / (1 + np.exp(-z))
print("z =", round(z, 3), " p (by hand) =", round(p_hand, 3),
      " p (model) =", round(model.predict_proba(case)[0, 1], 3))

**Python note.** `scaler.mean_` and `scaler.scale_` are the mean and the standard deviation that the scaler learned from the training set. `case.to_numpy()[0]` turns the one-row table into an array of ten numbers. `np.sum(a * b)` multiplies two arrays element by element and adds the products, which is the sum $\sum_j \beta_j x_j$ of the formula.

**Exercise A.** Patient VC-0246 has atypical angina (`cp_2.0 = 1`). What is his probability of CAD according to `model` if the chest pain is typical angina instead, with zeros in the three `cp` columns? Round to three decimals.

In [ ]:
data = exercise_table("A", X=X, X_test=X_test, y=y)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
typical = case.copy()
typical["cp_2.0"] = 0
answer = round(model.predict_proba(typical)[0, 1], 3)
teach.check("Exercise A", answer)

**Observe and reflect.** In the Cleveland data, the patients without chest pain (`cp_4.0 = 1`) have the highest odds of disease, and typical angina, the reference, has lower odds. How can a referral pattern explain this? What does it mean for using the model in a clinic that sees a different mix of patients?

**Advanced (optional).** Fit the same pipeline with `LogisticRegression(C=0.05, max_iter=1000)`. A small `C` means strong *regularisation*: The coefficients shrink toward zero. Compare the two tables of odds ratios and the test AUC of the part «Measuring a model» below.

In [ ]:
# Advanced (optional)
strong = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=1000)).fit(X_train, y_train)
pd.DataFrame({"C=1": np.exp(lr.coef_[0]),
              "C=0.05": np.exp(strong.named_steps["logisticregression"].coef_[0])},
             index=X.columns).round(2)

#### Measuring a model: Confusion matrix and ROC curve

*Lecture page: [Measuring a model: Confusion matrix and ROC curve](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#measuring-a-model-confusion-matrix-and-roc-curve)*

The test set holds 76 patients that the model has never seen. A threshold turns each probability into a decision, and the decisions are counted against the truth in a *confusion matrix*. Moving the threshold trades sensitivity against specificity, and the *receiver operating characteristic (ROC) curve* shows all thresholds at once.

**From the formulas to the code.** With the counts `tp`, `fp`, `tn` and `fn`, sensitivity is $Se = TP/(TP+FN)$, specificity $Sp = TN/(TN+FP)$, the positive predictive value $PPV = TP/(TP+FP)$ and the negative predictive value $NPV = TN/(TN+FN)$. `roc_curve` returns the false positive rate $1 - Sp$, the true positive rate $Se$ and the thresholds, and `roc_auc_score` the area under the curve (AUC). Youden's index is $J = Se + Sp - 1$.

**Start cell.** For the section «Measuring a model: Confusion matrix and ROC curve», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Measuring a model: Confusion matrix and ROC curve»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week02 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, mean_absolute_error,
                             precision_recall_curve, roc_auc_score, roc_curve,
                             silhouette_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Loading the Cleveland data from one of three sources
DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
FILE = "processed.cleveland.data"
URL = vclinic.COURSE_RAW + "data/uci-heart-disease/" + FILE   # the copy of the UCI file in the course repository
if DATA_SOURCE == "library":
    # ucimlrepo, the Python library of the UCI Machine Learning Repository, downloads the data set by its number
    vclinic.install("ucimlrepo")
    from ucimlrepo import fetch_ucirepo
    uci = fetch_ucirepo(id=45)
    heart = pd.concat([uci.data.features, uci.data.targets], axis=1)[COLUMNS]
elif DATA_SOURCE == "web":
    # pandas reads the file straight from its web address
    heart = pd.read_csv(URL, header=None, names=COLUMNS, na_values="?")
else:
    # the file on this computer: download it first with the link in the note below this cell
    path = vclinic.local_file(FILE)   # Jupyter looks in the folder of this notebook; Colab opens an upload window
    heart = pd.read_csv(path, header=None, names=COLUMNS, na_values="?")
heart["disease"] = (heart["num"] > 0).astype(int)   # 1 when a major vessel is narrowed by more than 50 percent

# The table of the model: one row per patient, numbers only
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
data = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"],
                      drop_first=True, dtype=int)
X = data.drop(columns="disease")
y = data["disease"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

# The model: standardisation and logistic regression in one pipeline
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
p_test = model.predict_proba(X_test)[:, 1]

**On the lecture page:** [Worked example: The test set of the mini project at the threshold 0.5](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#worked-example-the-test-set-of-the-mini-project-at-the-threshold-0-5). The two cells below compute the same confusion matrix, the four measures and Youden's index.

In [ ]:
# Confusion matrix at the threshold 0.5
pred = (p_test >= 0.5).astype(int)
print(pd.crosstab(y_test.to_numpy(), pred, rownames=["truth"], colnames=["model"]))
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f"TP={tp} FP={fp} TN={tn} FN={fn}")
print("Se =", round(tp / (tp + fn), 3), " Sp =", round(tn / (tn + fp), 3),
      " PPV =", round(tp / (tp + fp), 3), " NPV =", round(tn / (tn + fn), 3))

**Python note.** `(p_test >= 0.5)` gives `True` or `False` for each patient, and `.astype(int)` turns them into 1 and 0. `pd.crosstab` counts the pairs of truth and decision. `confusion_matrix(...).ravel()` returns the four counts in the order TN, FP, FN, TP, and the line unpacks them into four names. An *f-string* puts values inside braces into the text.

In [ ]:
# ROC curve, AUC and Youden's index
fpr, tpr, thresholds = roc_curve(y_test, p_test)
auc = roc_auc_score(y_test, p_test)
best = np.argmax(tpr - fpr)
print("AUC =", round(auc, 3))
print("Youden: threshold", round(thresholds[best], 3), " Se", round(tpr[best], 3),
      " Sp", round(1 - fpr[best], 3), " J", round(tpr[best] - fpr[best], 3))
fig, ax = plt.subplots(figsize=(4.8, 4.6))
ax.plot(fpr, tpr, color="#9B1B1B", lw=2, label=f"logistic regression (AUC {auc:.3f})")
ax.plot([0, 1], [0, 1], color="#888888", ls="--", lw=1, label="chance")
ax.scatter(fpr[best], tpr[best], color="#1F5F8B", zorder=3, label="Youden")
ax.set_xlabel("1 − specificity (false positive rate)")
ax.set_ylabel("Sensitivity (true positive rate)")
ax.legend(loc="lower right")
plt.show()

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-c135)*

**Python note.** `roc_curve` tries every useful threshold and returns three arrays of the same length. `np.argmax(tpr - fpr)` gives the position of the largest value of $Se - (1 - Sp) = J$, and that position picks the matching threshold.

**Explore.** Move the threshold. The confusion matrix, the four measures and the point on the ROC curve change together. A low threshold suits a test that should miss nobody, a high one a test that should start an invasive procedure only when it is fairly sure.

In [ ]:
# Explore: the threshold
def show_threshold(threshold=0.5):
    pred = (p_test >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    se, sp = tp / (tp + fn), tn / (tn + fp)
    ppv = tp / (tp + fp) if tp + fp else float("nan")
    fig, ax = plt.subplots(figsize=(4.4, 4.2))
    ax.plot(fpr, tpr, color="#9B1B1B", lw=2)
    ax.plot([0, 1], [0, 1], color="#888888", ls="--", lw=1)
    ax.scatter(1 - sp, se, s=60, color="#1F5F8B", zorder=3)
    ax.set_xlabel("1 − Sp")
    ax.set_ylabel("Se")
    ax.set_title(f"threshold {threshold:.2f}")
    plt.show()
    print(f"TP={tp} FP={fp} TN={tn} FN={fn} | Se={se:.3f} Sp={sp:.3f} PPV={ppv:.3f}")


try:
    from ipywidgets import interact, FloatSlider
    interact(show_threshold, threshold=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05,
                                                   description="threshold"))
except ImportError:
    for value in (0.3, 0.5, 0.7):
        show_threshold(value)

**Three thresholds, three clinical uses.** *Lecture page: [Interpreting sensitivity and specificity](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#interpreting-sensitivity-and-specificity)*

A low threshold makes the model sensitive, so that a negative result can rule CAD out. A high threshold makes it specific, so that a positive result can rule CAD in. The cell computes the four measures of the lecture at the thresholds 0.15, 0.5 and 0.7.

**On the lecture page:** [Worked example: Three thresholds for the model of the mini project](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#worked-example-three-thresholds-for-the-model-of-the-mini-project). The cell below computes the same table.

In [ ]:
# Sensitivity, specificity and predictive values at three thresholds
rows = []
for t in [0.15, 0.5, 0.7]:
    tn, fp, fn, tp = confusion_matrix(y_test, (p_test >= t).astype(int)).ravel()
    rows.append({"threshold": t, "TP": tp, "FN": fn, "TN": tn, "FP": fp,
                 "Se": tp / (tp + fn), "Sp": tn / (tn + fp), "PPV": tp / (tp + fp), "NPV": tn / (tn + fn)})
pd.DataFrame(rows).round(3)

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-c140)*

**Python note.** Each pass of the loop builds one dictionary, one row of the table, and `rows.append` adds it to the list. `pd.DataFrame(rows)` turns the list of dictionaries into a table whose columns are the keys.

**Exercise B.** What is the AUC of `model` on the test set? Round to three decimals.

In [ ]:
data = exercise_table("B", X_test=X_test, model=model, y_test=y_test)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 3)
teach.check("Exercise B", answer)

**Exercise C.** What is the sensitivity of `model` on the test set at the threshold 0.5? Round to three decimals.

In [ ]:
data = exercise_table("C", X_test=X_test, model=model, y_test=y_test)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
pred = (model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
answer = round(tp / (tp + fn), 3)
teach.check("Exercise C", answer)

**Observe and reflect.** An AUC of 0.93 sounds excellent. Which questions should a cardiologist still ask before using the model for decisions about angiography? Think of the size of the test set, of where the patients came from, and of what a false negative costs compared with a false positive.

**Stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#decision-trees-and-random-forests) and continue reading at «Comparing a decision tree and a random forest». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 2: Testing a model honestly

*Lecture page: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#stop-2)*

A single test set of 76 patients gives an uncertain estimate, and some models look better than they are. In this stop the test is repeated with cross-validation and a random forest is tried. Then the work turns to a rare outcome of the virtual clinic: death or a hospitalisation for heart failure within 12 months.

**At the end of this stop:** The cross-validated AUC of the logistic regression (0.871), the test AUC of a random forest (0.911) and the sensitivity of a model of the rare 12-month event trained with class weights (0.65).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 2 · Python basics

*Lecture page: [Training and testing: Cross-validation and data leakage](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#training-and-testing-cross-validation-and-data-leakage)*

The ready application of this stop fits a model again in each of five folds, compares several models in one table, draws random numbers and writes small functions. The examples below introduce these constructs with numbers from cardiology: a loop with `enumerate`, a table built row by row from dictionaries, random numbers with a seed value and a function with a default value.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 2 · Python basics», the cell below repeats the setup of the course package and of the packages of the week, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

In [ ]:
# Python practice: a small example for learning the language
fold_auc = [0.91, 0.84, 0.88, 0.79, 0.93]   # the AUC of the five folds of a cross-validation
for fold, auc in enumerate(fold_auc, start=1):
    print("fold", fold, "AUC", auc)
print("mean", round(np.mean(fold_auc), 3), "| standard deviation", round(np.std(fold_auc), 3))

**Python note.** `enumerate(fold_auc, start=1)` gives, in each pass of the loop, the number of the item and the item itself, so `fold` counts 1, 2, 3, 4 and 5 while `auc` takes the five values. `np.mean` and `np.std` work on a list directly. `np.std` divides by the number of values; the standard deviation that the ready application reports after a cross-validation is computed in the same way.

In [ ]:
# Python practice: a small example for learning the language
results = {"decision tree": 0.80, "random forest": 0.91, "logistic regression": 0.92}
rows = []
for name, auc in results.items():
    rows.append({"model": name, "AUC": auc, "above 0.85": auc > 0.85})
table = pd.DataFrame(rows)
table.sort_values("AUC", ascending=False)

**Python note.** A *dictionary*, written with braces, keeps values under names: `results["random forest"]` is 0.91. `.items()` gives the name and the value together in each pass of the loop. Each pass builds one dictionary, one row of the table, and `rows.append` adds it to the list `rows`. `pd.DataFrame(rows)` turns the list of dictionaries into a table whose columns are the names of the dictionaries, and `sort_values("AUC", ascending=False)` puts the best model first.

In [ ]:
# Python practice: a small example for learning the language
rng = np.random.default_rng(RANDOM_STATE)
noise = rng.normal(size=(5, 3))   # 5 patients, 3 random variables
print(noise.round(2))
again = np.random.default_rng(RANDOM_STATE).normal(size=(5, 3))
print("the same numbers again:", np.array_equal(noise, again))

**Python note.** `np.random.default_rng(RANDOM_STATE)` creates a generator of random numbers. With the same seed value it produces the same numbers at every run, which is why every split and every model of the course uses `RANDOM_STATE`. `rng.normal(size=(5, 3))` draws numbers of the standard normal distribution into an array of 5 rows and 3 columns, a table of random variables without any information about the patients.

In [ ]:
# Python practice: a small example for learning the language
def ppv(sensitivity, specificity, prevalence=0.05):
    """Positive predictive value from the sensitivity, the specificity and the prevalence (Bayes' theorem)."""
    true_pos = sensitivity * prevalence
    false_pos = (1 - specificity) * (1 - prevalence)
    return true_pos / (true_pos + false_pos)


print("PPV, prevalence 5 percent:", round(ppv(0.80, 0.95), 3))
print("PPV, prevalence 30 percent:", round(ppv(0.80, 0.95, prevalence=0.30), 3))

**Python note.** `def` defines a function: Its name, its *parameters* in parentheses and, indented below, the lines that it runs. The text in triple quotes right under `def`, the *docstring*, says what the function does. `prevalence=0.05` is a default value, used when the call does not give the parameter, as in the first call. `return` gives the result back to the line that called the function. With the same test, the PPV rises from 0.457 to 0.873 when the prevalence rises from 5 to 30 percent.

**Exercise 2.1.** Five folds of a cross-validation give the AUC values 0.88, 0.91, 0.85, 0.90 and 0.86. What is their standard deviation with `np.std`? Round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(np.std([0.88, 0.91, 0.85, 0.90, 0.86]), 3)
teach.check("Exercise 2.1", answer)

**Exercise 2.2.** Use the function `ppv` above: What is the PPV of a test with a sensitivity of 0.90 and a specificity of 0.80 at a prevalence of 10 percent? Round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(ppv(0.90, 0.80, prevalence=0.10), 3)
teach.check("Exercise 2.2", answer)

**«Stop 2 · Python basics» is complete.** Next comes «Stop 2 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#unsupervised-learning-k-means-and-phenogroups) and continue reading at «Finding subgroups of patients with k-means».

### Stop 2 · Prompt and application

This part has three prompts: «Five-fold cross-validation», «A random forest» and «A rare outcome». The repair prompt and the prompt for an error message are in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md) and in the prompts of Stop 1.

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Start cell.** For «Stop 2 · Prompt and application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic.datasets import load_cleveland

DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
heart = load_cleveland(source=DATA_SOURCE)
_FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
_coded = pd.get_dummies(heart[_FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
_X, _y = _coded.drop(columns="disease"), _coded["disease"]
_Xtr, _Xte, _ytr, _yte = train_test_split(_X, _y, test_size=0.25, stratify=_y, random_state=RANDOM_STATE)
_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(_Xtr, _ytr)
_CP = {1: "typical angina", 2: "atypical angina", 3: "non-anginal pain", 4: "asymptomatic"}


def _cad_table():
    """The Cleveland patients with the eight variables of the model and the outcome."""
    return pd.DataFrame({"age": heart["age"].astype(int), "sex": heart["sex"].astype(int),
                         "chest pain": heart["cp"].map(_CP), "resting blood pressure": heart["trestbps"].astype(int),
                         "cholesterol": heart["chol"].astype(int), "highest heart rate": heart["thalach"].astype(int),
                         "exercise angina": heart["exang"].astype(int), "ST depression": heart["oldpeak"],
                         "CAD": heart["disease"].astype(int)})
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, mean_absolute_error,
                             precision_recall_curve, roc_auc_score, roc_curve,
                             silhouette_score)

#### Prompt 2.1: Five-fold cross-validation

**Goal.** Test the logistic regression of Stop 1 five times, each time on another fifth of the patients, and take the mean AUC.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants a more reliable estimate of the AUC of a model of coronary artery disease than one test set of 76 patients can give, from 303 patients of the Cleveland Clinic.  
> Data: The notebook holds a table named "data". Each row is a patient: "age" gives the age in years, and "sex" is 1 for men and 0 for women. The column "chest pain" says "typical angina", "atypical angina", "non-anginal pain" or "asymptomatic". "resting blood pressure" gives the resting systolic blood pressure in mm Hg, "cholesterol" the serum cholesterol in mg/dL and "highest heart rate" the highest heart rate of the exercise test in beats per minute. "exercise angina" is 1 when angina occurred during exercise and 0 otherwise, and "ST depression" gives the ST depression during exercise compared with rest in mm. "CAD" is 1 when at least one major coronary artery was narrowed by more than 50 percent of its diameter at angiography, and 0 otherwise. The column "fold" gives the fold of the patient, 1 to 5; the share of patients with CAD is about the same in every fold.  
> Task: Turn the type of chest pain into three yes/no columns, with typical angina as the reference. Use the eight variables other than "CAD" and "fold" as the inputs. For each fold in turn, use this fold as the test set and the other four folds as the training set: Standardise the variables with the training patients, fit a logistic regression and compute the AUC on the test fold. Print the five AUC values and their mean.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the mean AUC of the five folds with three decimals and keep it under the name "answer".  
> Test: Five values must be printed, each between 0.5 and 1.  

In [ ]:
# Data of Prompt 2.1: the table "data" that the prompt describes
data = _cad_table()
data["fold"] = 0
for _k, (_, _rows) in enumerate(StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE).split(_X, _y), start=1):
    data.iloc[_rows, -1] = _k
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.1
teach.check_step("Prompt 2.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.1 (run it only if the code of the assistant does not pass the check)
data = _cad_table()
data["fold"] = 0
for _k, (_, _rows) in enumerate(StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE).split(_X, _y), start=1):
    data.iloc[_rows, -1] = _k
X_table = data[["age", "sex", "resting blood pressure", "cholesterol",
                    "highest heart rate", "exercise angina", "ST depression"]].copy()
for pain in ["atypical angina", "non-anginal pain", "asymptomatic"]:
    X_table[pain] = (data["chest pain"] == pain).astype(int)
outcome = data["CAD"]
values = []
for fold in range(1, 6):
    test_rows = data["fold"] == fold
    lr_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    lr_model.fit(X_table[~test_rows], outcome[~test_rows])
    probabilities = lr_model.predict_proba(X_table[test_rows])[:, 1]
    values.append(roc_auc_score(outcome[test_rows], probabilities))
print("AUC of the five folds:", np.round(values, 3))
answer = round(float(np.mean(values)), 3)
print("Mean AUC:", answer)
teach.check_step("Prompt 2.1", "answer", globals())

**What the result shows.** The mean is lower than the AUC of the single test set of Prompt 1.1, and the five values spread widely. On 303 patients, the result depends noticeably on which patients happen to be in the test set.

#### Prompt 2.2: A random forest

**Goal.** Train a more flexible model on the same patients and compare its test AUC with the logistic regression.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: For 303 patients of the Cleveland Clinic, the cardiologist wants to know whether a random forest, a more flexible model, predicts coronary artery disease better than a logistic regression.  
> Data: The notebook holds a table named "data". Each row is a patient: "age" gives the age in years, and "sex" is 1 for men and 0 for women. The column "chest pain" says "typical angina", "atypical angina", "non-anginal pain" or "asymptomatic". "resting blood pressure" gives the resting systolic blood pressure in mm Hg, "cholesterol" the serum cholesterol in mg/dL and "highest heart rate" the highest heart rate of the exercise test in beats per minute. "exercise angina" is 1 when angina occurred during exercise and 0 otherwise, and "ST depression" gives the ST depression during exercise compared with rest in mm. "CAD" is 1 when at least one major coronary artery was narrowed by more than 50 percent of its diameter at angiography, and 0 otherwise. The column "set" says "training" or "test"; the test set holds 76 patients.  
> Task: Turn the type of chest pain into three yes/no columns, with typical angina as the reference. Use the eight variables other than "CAD" and "set" as the inputs. Train a random forest on the training set. Make the random choices the same in every run by using the number 2026 as the starting value. Compute the AUC of the probabilities that the forest gives to the test patients.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the test AUC of the forest with three decimals and keep it under the name "answer".  
> Test: The AUC must lie between 0.5 and 1. For comparison, also compute the test AUC of a logistic regression on the same sets and print the two values side by side.  

In [ ]:
# Data of Prompt 2.2: the table "data" that the prompt describes
data = _cad_table()
data["set"] = np.where(data.index.isin(_Xte.index), "test", "training")
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.2
teach.check_step("Prompt 2.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.2 (run it only if the code of the assistant does not pass the check)
data = _cad_table()
data["set"] = np.where(data.index.isin(_Xte.index), "test", "training")
X_table = data[["age", "sex", "resting blood pressure", "cholesterol",
                    "highest heart rate", "exercise angina", "ST depression"]].copy()
for pain in ["atypical angina", "non-anginal pain", "asymptomatic"]:
    X_table[pain] = (data["chest pain"] == pain).astype(int)
outcome = data["CAD"]
test_rows = data["set"] == "test"
forest = RandomForestClassifier(n_estimators=300, random_state=2026)
forest.fit(X_table[~test_rows], outcome[~test_rows])
answer = round(roc_auc_score(outcome[test_rows], forest.predict_proba(X_table[test_rows])[:, 1]), 3)
print("Test AUC of the forest:", answer)
teach.check_step("Prompt 2.2", "answer", globals())

**What the result shows.** The forest does not beat the logistic regression on these patients. Flexible models need many patients; on a small table, a simple and readable model is often just as good.

#### Prompt 2.3: A rare outcome

**Goal.** Train a model of an event within 12 months, which only about 3 of 100 patients have, and see how class weights change what the model finds.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Among the 2000 patients of the virtual clinic, the cardiologist wants a model that predicts a rare event: death or a hospitalisation for heart failure within the first 12 months.  
> Data: The notebook holds a table named "data". Each row is a patient: "age" (years), "eGFR" (estimated glomerular filtration rate, mL/min/1.73 m²), "LVEF" (left ventricular ejection fraction, %), "NT-proBNP" (pg/mL), "AF" (atrial fibrillation, 1 for yes) and "diabetes" (1 for yes). "event" is 1 for death or a hospitalisation for heart failure within the first 12 months, and 0 otherwise. The column "set" says "training" or "test"; the test set holds 600 patients.  
> Task: Use the natural logarithm of NT-proBNP in place of NT-proBNP. Standardise the six variables with the training patients and fit a logistic regression on the training set. Balance the classes in training: The few patients with an event must carry as much weight in total as all the patients without one. Call a test patient positive when the probability is 0.5 or more, and compute the sensitivity, TP / (TP + FN).  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the sensitivity with three decimals and keep it under the name "answer".  
> Test: The test set must hold 600 patients, 20 of them with an event. The same model trained without balancing the classes must find none of these events at the threshold 0.5.  

In [ ]:
# Data of Prompt 2.3: the table "data" that the prompt describes
_e = registry[["age", "egfr", "lvef", "ntprobnp", "af", "diabetes", "event_12m"]]
_e_train, _e_test = train_test_split(_e, test_size=0.3, stratify=_e["event_12m"], random_state=RANDOM_STATE)
data = _e.rename(columns={"age": "age", "egfr": "eGFR", "lvef": "LVEF", "ntprobnp": "NT-proBNP", "af": "AF",
                          "diabetes": "diabetes", "event_12m": "event"})
data["set"] = np.where(data.index.isin(_e_test.index), "test", "training")
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.3
teach.check_step("Prompt 2.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.3 (run it only if the code of the assistant does not pass the check)
_e = registry[["age", "egfr", "lvef", "ntprobnp", "af", "diabetes", "event_12m"]]
_e_train, _e_test = train_test_split(_e, test_size=0.3, stratify=_e["event_12m"], random_state=RANDOM_STATE)
data = _e.rename(columns={"age": "age", "egfr": "eGFR", "lvef": "LVEF", "ntprobnp": "NT-proBNP", "af": "AF",
                          "diabetes": "diabetes", "event_12m": "event"})
data["set"] = np.where(data.index.isin(_e_test.index), "test", "training")
inputs = data[["age", "eGFR", "LVEF", "NT-proBNP", "AF", "diabetes"]].assign(**{"NT-proBNP": np.log(data["NT-proBNP"])})
test_rows = data["set"] == "test"
event = data["event"]
weighted = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
weighted.fit(inputs[~test_rows], event[~test_rows])
positive = weighted.predict_proba(inputs[test_rows])[:, 1] >= 0.5
true_events = event[test_rows].to_numpy() == 1
tp = int((positive & true_events).sum())
fn = int((~positive & true_events).sum())
answer = round(tp / (tp + fn), 3)
print("Events in the test set:", int(true_events.sum()), " sensitivity =", answer)
teach.check_step("Prompt 2.3", "answer", globals())

**What the result shows.** Without class weights, a model that predicts no event for everybody is already 97 percent accurate and finds none of the events. The weights make the rare events count more in training; the price is a low positive predictive value, many alarms for few events.

**«Stop 2 · Prompt and application» is complete.** Next comes «Stop 2 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#unsupervised-learning-k-means-and-phenogroups) and continue reading at «Finding subgroups of patients with k-means».

### Stop 2 · Ready application

This part tests the model honestly: five-fold cross-validation of the whole pipeline and a leakage experiment on pure noise, a decision tree and a random forest with the permutation importance of their variables, the model of coronary artery disease in the registry of the virtual clinic against its true effects, and the rare event within 12 months with class weights, precision and recall and SMOTE.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 2 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week02 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, mean_absolute_error,
                             precision_recall_curve, roc_auc_score, roc_curve,
                             silhouette_score)
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Loading the Cleveland data from one of three sources
DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
FILE = "processed.cleveland.data"
URL = vclinic.COURSE_RAW + "data/uci-heart-disease/" + FILE   # the copy of the UCI file in the course repository
if DATA_SOURCE == "library":
    # ucimlrepo, the Python library of the UCI Machine Learning Repository, downloads the data set by its number
    vclinic.install("ucimlrepo")
    from ucimlrepo import fetch_ucirepo
    uci = fetch_ucirepo(id=45)
    heart = pd.concat([uci.data.features, uci.data.targets], axis=1)[COLUMNS]
elif DATA_SOURCE == "web":
    # pandas reads the file straight from its web address
    heart = pd.read_csv(URL, header=None, names=COLUMNS, na_values="?")
else:
    # the file on this computer: download it first with the link in the note below this cell
    path = vclinic.local_file(FILE)   # Jupyter looks in the folder of this notebook; Colab opens an upload window
    heart = pd.read_csv(path, header=None, names=COLUMNS, na_values="?")
heart["disease"] = (heart["num"] > 0).astype(int)   # 1 when a major vessel is narrowed by more than 50 percent

# The table of the model: one row per patient, numbers only
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
data = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"],
                      drop_first=True, dtype=int)
X = data.drop(columns="disease")
y = data["disease"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

#### Cross-validation and data leakage

*Lecture page: [Training and testing: Cross-validation and data leakage](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#training-and-testing-cross-validation-and-data-leakage)*

One test set of 76 patients gives one number, and another split would give another. *K-fold cross-validation* splits the patients into $K$ parts, trains on $K-1$ of them, tests on the remaining one and repeats this $K$ times. The mean of the $K$ results is a more stable estimate, and their spread shows how much it moves. Every step that learns from the data, scaling and selection of variables included, must be repeated inside each fold, or information from the test patients *leaks* into the model.

**From the formulas to the code.** The estimate is $\overline{AUC} = \frac{1}{K}\sum_{k=1}^{K} AUC_k$. `StratifiedKFold(5, shuffle=True, random_state=...)` makes the five folds with the same share of disease in each, and `cross_val_score` returns the five $AUC_k$.

**On the lecture page:** [Worked example: Five folds of the Cleveland data](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#worked-example-five-folds-of-the-cleveland-data). The cell below repeats the same cross-validation.

In [ ]:
# Five-fold cross-validation of the whole pipeline
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
print("AUC of the five folds:", scores.round(3))
print("Mean", round(scores.mean(), 3), " SD", round(scores.std(), 3))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-c153)*

**Python note.** `cross_val_score` takes the unfitted pipeline and the whole `X` and `y`. In each fold it fits a fresh copy of the pipeline on four parts, so the scaler also learns from the training folds only, and it scores the fifth part. `scoring="roc_auc"` asks for the AUC.

**On the lecture page:** [Worked example: An AUC of 0.73 from pure noise](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#worked-example-an-auc-of-0-73-from-pure-noise). The cell below repeats the experiment, once with the selection before the folds and once inside them.

In [ ]:
# Leakage on pure noise: 500 random variables and no real signal
rng = np.random.default_rng(RANDOM_STATE)
noise = rng.normal(size=(len(y), 500))

# Wrong: choose the 10 "best" variables on all patients, then cross-validate
chosen = SelectKBest(f_classif, k=10).fit_transform(noise, y)
leaky = cross_val_score(LogisticRegression(max_iter=1000), chosen, y, cv=cv, scoring="roc_auc")

# Right: the selection is a step of the pipeline and is repeated in each fold
honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=10), LogisticRegression(max_iter=1000)),
                         noise, y, cv=cv, scoring="roc_auc")
print("Selection before cross-validation: AUC", round(leaky.mean(), 3))
print("Selection inside the pipeline:     AUC", round(honest.mean(), 3))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-c155)*

**Python note.** `rng.normal(size=(303, 500))` draws a table of random numbers with 303 rows and 500 columns. `SelectKBest(f_classif, k=10)` keeps the 10 columns that differ most between the two groups. On pure noise some columns differ by chance, and choosing them with all patients lets the test folds take part in the choice.

**Exercise D.** What is the mean cross-validated AUC of `pipe` over the five folds of `cv`? Round to three decimals.

In [ ]:
data = exercise_table("D", X=X, cv=cv, y=y)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean(), 3)
teach.check("Exercise D", answer)

**Observe and reflect.** The leaky version finds an AUC of about 0.7 in data that contain no information at all. Published models are sometimes built this way. Which sentence in the methods section of a paper would make you suspect it, and which sentence would reassure you?

**Advanced (optional).** Repeat the cross-validation ten times with different shuffles, using `RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)` from `sklearn.model_selection`, and plot the histogram of the 50 AUC values.

In [ ]:
# Advanced (optional)
from sklearn.model_selection import RepeatedStratifiedKFold
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
many = cross_val_score(pipe, X, y, cv=rcv, scoring="roc_auc")
plt.hist(many, bins=12, color="#1F5F8B")
plt.xlabel("AUC")
plt.ylabel("Number of folds")
plt.show()
print(round(many.mean(), 3), round(many.std(), 3))

#### Decision trees and random forests

*Lecture page: [Comparing a decision tree and a random forest](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#decision-trees-and-random-forests)*

A decision tree asks one yes or no question at a time, like a clinical algorithm. At each node it chooses the question that makes the two groups purest. A random forest grows hundreds of trees, each on a bootstrap sample of the patients and with a random subset of the variables at each node, and averages their probabilities.

**From the formulas to the code.** The Gini impurity of a node with the class shares $p_0$ and $p_1$ is $G = 1 - (p_0^2 + p_1^2)$. A split is scored by the impurity of its two children, weighted by their sizes. `DecisionTreeClassifier(max_depth=3)` limits the tree to three levels of questions, and `RandomForestClassifier(n_estimators=300)` averages 300 trees.

**Start cell.** For the section «Decision trees and random forests», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Decision trees and random forests»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week02 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, mean_absolute_error,
                             precision_recall_curve, roc_auc_score, roc_curve,
                             silhouette_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Loading the Cleveland data from one of three sources
DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
FILE = "processed.cleveland.data"
URL = vclinic.COURSE_RAW + "data/uci-heart-disease/" + FILE   # the copy of the UCI file in the course repository
if DATA_SOURCE == "library":
    # ucimlrepo, the Python library of the UCI Machine Learning Repository, downloads the data set by its number
    vclinic.install("ucimlrepo")
    from ucimlrepo import fetch_ucirepo
    uci = fetch_ucirepo(id=45)
    heart = pd.concat([uci.data.features, uci.data.targets], axis=1)[COLUMNS]
elif DATA_SOURCE == "web":
    # pandas reads the file straight from its web address
    heart = pd.read_csv(URL, header=None, names=COLUMNS, na_values="?")
else:
    # the file on this computer: download it first with the link in the note below this cell
    path = vclinic.local_file(FILE)   # Jupyter looks in the folder of this notebook; Colab opens an upload window
    heart = pd.read_csv(path, header=None, names=COLUMNS, na_values="?")
heart["disease"] = (heart["num"] > 0).astype(int)   # 1 when a major vessel is narrowed by more than 50 percent

# The table of the model: one row per patient, numbers only
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
data = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"],
                      drop_first=True, dtype=int)
X = data.drop(columns="disease")
y = data["disease"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

**On the lecture page:** [Worked example: The first question of the Cleveland tree](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#worked-example-the-first-question-of-the-cleveland-tree). The cell below grows the same tree and draws it.

In [ ]:
# A tree with three levels of questions
tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(X_train, y_train)
print("Test AUC of the tree:", round(roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]), 3))
fig, ax = plt.subplots(figsize=(13, 5.5))
plot_tree(tree, feature_names=list(X.columns), class_names=["no CAD", "CAD"],
          filled=True, impurity=True, fontsize=8, ax=ax)
plt.show()

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-c166)*

**Python note.** `plot_tree` draws each node with its question, its Gini impurity, the number of patients (`samples`) and the count of each class (`value`). The left branch of a node holds the patients for whom the question is true.

**Exercise E.** Compute the Gini impurity of the root of the tree, before any question, from the counts of the two classes in `y_train`. Round to three decimals.

In [ ]:
data = exercise_table("E", y_train=y_train)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n0, n1 = (y_train == 0).sum(), (y_train == 1).sum()
p0, p1 = n0 / (n0 + n1), n1 / (n0 + n1)
answer = round(1 - (p0 ** 2 + p1 ** 2), 3)
teach.check("Exercise E", answer)

In [ ]:
# A random forest and the permutation importance of its variables
forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train, y_train)
print("Test AUC of the forest:", round(roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]), 3))
imp = permutation_importance(forest, X_test, y_test, scoring="roc_auc", n_repeats=20,
                             random_state=RANDOM_STATE)
order = np.argsort(imp.importances_mean)
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(X.columns[order], imp.importances_mean[order], xerr=imp.importances_std[order], color="#1F5F8B")
ax.set_xlabel("Drop in AUC when the variable is shuffled")
plt.show()

**Python note.** *Permutation importance* shuffles one column of the test set at a time and measures how much the AUC drops. A large drop means that the forest relies on that variable. `np.argsort` returns the positions that would sort the values, so the bars appear from the smallest to the largest.

In [ ]:
# The three models on the same test set and the same five folds
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
models = {"decision tree, three levels": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
          "random forest, 300 trees": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
          "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))}
rows = []
for name, m in models.items():
    p = m.fit(X_train, y_train).predict_proba(X_test)[:, 1]
    folds = cross_val_score(m, X, y, cv=cv, scoring="roc_auc")
    rows.append({"model": name, "AUC on the test set": roc_auc_score(y_test, p),
                 "AUC in cross-validation, mean": folds.mean(),
                 "SD over the folds": folds.std()})
pd.DataFrame(rows).round(3)

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-c173)*

**Python note.** The dictionary `models` holds three models under their names, and `.items()` gives the name and the model in each pass of the loop. `m.fit(...)` returns the fitted model itself, so `.predict_proba` can follow on the same line. `cross_val_score` fits a fresh copy of the model in each of the five folds of `cv`, the same folds as in the part «Cross-validation and data leakage» above.

**Observe and reflect.** The single tree is easy to read but reaches a lower AUC than the forest, and the forest is accurate but hard to read. For a decision at the bedside, which would you prefer, and what would you need to see before trusting the forest?

#### The risk outpatient clinic: Coronary artery disease in the registry

*Lecture page: [The virtual clinic, stage 2: The risk outpatient clinic opens](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#the-virtual-clinic-stage-2-the-risk-outpatient-clinic-opens)*

The virtual clinic grows to 2000 patients and opens its risk outpatient clinic. The course generated the coronary artery disease of each patient from known log-odds effects, stored in `vclinic.TRUE_EFFECTS`. A logistic regression fitted to the registry should therefore find coefficients close to these values, which a model on real data can never show. `vclinic.add_followup` adds the echocardiographic details and the follow-up used in the parts that follow.

**From the formulas to the code.** The variables are coded as in `TRUE_EFFECTS`: age per 10 years above 60, $(LDL - 130)/40$ and $(HDL - 50)/15$, and 0 or 1 for the others. `LogisticRegression(C=np.inf)` fits the model without regularisation, so its `coef_` are the maximum likelihood estimates of the $\beta_j$.

**Start cell.** For the section «The risk outpatient clinic: Coronary artery disease in the registry», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «The risk outpatient clinic: Coronary artery disease in the registry»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week02 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LinearRegression, LogisticRegression

In [ ]:
# The registry with follow-up: 2000 patients
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
print(registry.shape)
print(registry["main_diagnosis"].value_counts().to_string())


def risk_table(df):
    """The variables coded as in vclinic.TRUE_EFFECTS."""
    return pd.DataFrame({
        "age_per_10y": (df["age"] - 60) / 10,
        "male": (df["sex"] == "M").astype(int),
        "smoker": df["smoker"],
        "diabetes": df["diabetes"],
        "hypertension": df["hypertension"],
        "ldl_per_40": (df["ldl"] - 130) / 40,
        "hdl_per_15": (df["hdl"] - 50) / 15,
        "family_history": df["family_history"],
    })


Z = risk_table(registry)
cad_model = LogisticRegression(C=np.inf, max_iter=2000).fit(Z, registry["cad"])
truth = pd.Series(vclinic.TRUE_EFFECTS)
compare = pd.DataFrame({"learned": np.r_[cad_model.intercept_, cad_model.coef_[0]],
                        "true": truth[["intercept"] + list(Z.columns)].to_numpy()},
                       index=["intercept"] + list(Z.columns))
compare.round(2)

**Python note.** A *docstring*, the text in triple quotes right under `def`, describes what a function does. `np.r_[a, b]` joins two arrays into one. `truth[["intercept"] + list(Z.columns)]` puts the true values in the same order as the learned ones, so that the two columns of the table can be compared row by row.

**On the lecture page:** [Worked example: The probability of VC-0246 from the true effects of the clinic](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#worked-example-the-probability-of-vc-0246-from-the-true-effects-of-the-clinic). The cell below computes the same probability from the true effects of the clinic and from the model learned on the registry.

In [ ]:
# The case of the week, VC-0246, in the risk outpatient clinic
row = registry[registry["patient_id"] == "VC-0246"]
z_true = truth["intercept"] + (risk_table(row).iloc[0] * truth[Z.columns]).sum()
print(row[["age", "sex", "smoker", "hypertension", "diabetes", "ldl", "hdl", "family_history"]].to_string(index=False))
print("Probability of CAD, model:", round(cad_model.predict_proba(risk_table(row))[0, 1], 3),
      " true:", round(1 / (1 + np.exp(-z_true)), 3))
print("Coronary angiography in the registry (cad):", int(row["cad"].iloc[0]))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-c181)*

**Exercise F.** What log-odds coefficient does `cad_model` learn for `smoker`? Round to two decimals and compare it with `TRUE_EFFECTS["smoker"]`.

In [ ]:
data = exercise_table("F", registry=registry)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(cad_model.coef_[0][list(Z.columns).index("smoker")], 2)
teach.check("Exercise F", answer)

**Observe and reflect.** The learned coefficients are close to the truth but not equal to it, although the model is the right one. What would happen to the differences with 200 patients, and with 20000? VC-0246 has a probability of about 0.8 and a normal angiogram. Is the model wrong for him?

#### Rare outcomes: Imbalanced data and SMOTE

*Lecture page: [Dealing with imbalanced data (SMOTE)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#rare-outcomes-imbalanced-data)*

Death or hospitalisation for heart failure within 12 months happens to about 3 of 100 patients of the registry. With such a rare outcome, a model that always answers *no event* is right 97 times out of 100 and useless. This section shows why accuracy misleads, how class weights and SMOTE shift a model toward the rare class and why the positive predictive value depends on the prevalence.

**From the formulas to the code.** Bayes' theorem gives $PPV = \frac{Se \cdot \pi}{Se \cdot \pi + (1 - Sp)(1 - \pi)}$ for a prevalence $\pi$. `class_weight="balanced"` gives each class the weight $n/(2 n_k)$, so that the rare class counts as much as the common one. `precision_recall_curve` returns the PPV (*precision*) and the sensitivity (*recall*) at every threshold, and `average_precision_score` summarises the curve.

**Start cell.** For the section «Rare outcomes: Imbalanced data and SMOTE», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Rare outcomes: Imbalanced data and SMOTE»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, mean_absolute_error,
                             precision_recall_curve, roc_auc_score, roc_curve,
                             silhouette_score)
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# The registry with follow-up: 2000 patients
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))

In [ ]:
# The rare outcome: an event within 12 months
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)
EVENT_FEATURES = ["age", "egfr", "lvef", "ntprobnp", "af", "diabetes"]
Xe = registry[EVENT_FEATURES].assign(ntprobnp=np.log(registry["ntprobnp"]))
ye = registry["event_12m"]
Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    Xe, ye, test_size=0.3, stratify=ye, random_state=RANDOM_STATE)
print("Prevalence:", round(ye.mean(), 3), " events in the test set:", int(ye_test.sum()), "/", len(ye_test))
print("Accuracy of always answering no event:", round((ye_test == 0).mean(), 3))

rows = []
for weight in [None, "balanced"]:
    m = make_pipeline(StandardScaler(), LogisticRegression(class_weight=weight, max_iter=1000)).fit(Xe_train, ye_train)
    p = m.predict_proba(Xe_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(ye_test, (p >= 0.5).astype(int)).ravel()
    rows.append({"class_weight": str(weight), "AUC": roc_auc_score(ye_test, p),
                 "AP": average_precision_score(ye_test, p), "accuracy": (tp + tn) / len(ye_test),
                 "Se": tp / (tp + fn), "PPV": tp / (tp + fp) if tp + fp else np.nan})
pd.DataFrame(rows).round(3)

**Python note.** `.assign(ntprobnp=...)` returns a copy of the table in which one column is replaced, here by its logarithm, because NT-proBNP spans several orders of magnitude. `str(weight)` turns `None` into the text "None" for the table. `np.nan` marks a value that cannot be computed, a PPV without any positive decision.

**On the lecture page:** [Worked example: The same model in two populations](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#worked-example-the-same-model-in-two-populations). The cell below shows the same effect for the model of the 12-month event: its PPV against the prevalence.

In [ ]:
# Precision and recall, and the PPV as a function of the prevalence
balanced = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)).fit(Xe_train, ye_train)
precision, recall, _ = precision_recall_curve(ye_test, balanced.predict_proba(Xe_test)[:, 1])
prev = np.linspace(0.005, 0.5, 200)
se, sp = 0.85, 0.90
ppv = se * prev / (se * prev + (1 - sp) * (1 - prev))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(recall, precision, color="#9B1B1B")
ax[0].axhline(ye_test.mean(), color="#888888", ls="--", label="prevalence")
ax[0].set_xlabel("Recall (sensitivity)")
ax[0].set_ylabel("Precision (PPV)")
ax[0].legend()
ax[1].plot(prev * 100, ppv, color="#1F5F8B")
ax[1].set_xlabel("Prevalence (%)")
ax[1].set_ylabel("PPV")
ax[1].set_title("Se 0.85, Sp 0.90")
plt.show()

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-c191)*

**Python note.** `np.linspace(0.005, 0.5, 200)` makes 200 evenly spaced prevalences. The formula of Bayes' theorem then runs on all of them at once, because NumPy works element by element on arrays. `_` is a name for a returned value that is not needed, here the thresholds.

**Exercise G.** A screening model has a sensitivity of 0.85 and a specificity of 0.90. What is its PPV in a population with a prevalence of 5 percent? Round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
se, sp, prev = 0.85, 0.90, 0.05
answer = round(se * prev / (se * prev + (1 - sp) * (1 - prev)), 3)
teach.check("Exercise G", answer)

**Observe and reflect.** With class weights, the model finds about two thirds of the events, but only about one in ten of its alarms is a real event. Would such a model be useful to plan the follow-up visits of a heart failure clinic? Which threshold would you choose, and who should take part in that choice?

**SMOTE: synthetic patients of the rare class.** *Lecture page: [Dealing with imbalanced data (SMOTE)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#rare-outcomes-imbalanced-data)*

SMOTE places new patients of the rare class on the line between a real patient of that class and one of its $k$ nearest neighbours of the same class: $\mathbf{x}_{new} = \mathbf{x}_i + \lambda \, (\mathbf{x}_{nn} - \mathbf{x}_i)$ with $0 \le \lambda \le 1$. The function below writes the method out in a few lines. It works on the standardised training patients only, after the split, and the test patients stay untouched.

**On the lecture page:** [Worked example: One synthetic patient](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#worked-example-one-synthetic-patient). The cell below creates synthetic patients in the same way, for the training patients with an event.

In [ ]:
# SMOTE by hand, on the training patients only
def smote(X, y, k=5, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    X_min = X[y == 1]                                     # patients of the rare class
    n_new = int((y == 0).sum() - (y == 1).sum())          # enough for two classes of equal size
    _, idx = NearestNeighbors(n_neighbors=k + 1).fit(X_min).kneighbors(X_min)
    i = rng.integers(0, len(X_min), n_new)                # a random patient of the rare class
    j = idx[i, rng.integers(1, k + 1, n_new)]             # one of its k nearest neighbours
    lam = rng.random((n_new, 1))                          # lambda between 0 and 1
    X_new = X_min[i] + lam * (X_min[j] - X_min[i])
    return np.vstack([X, X_new]), np.concatenate([y, np.ones(n_new, dtype=int)])


scaler_e = StandardScaler().fit(Xe_train)
Z_train, Z_test = scaler_e.transform(Xe_train), scaler_e.transform(Xe_test)
Z_smote, y_smote = smote(Z_train, ye_train.to_numpy())
print("Training set before SMOTE:", len(ye_train), "patients,", int(ye_train.sum()), "events")
print("Training set after SMOTE:", len(y_smote), "patients,", int(y_smote.sum()), "events")

rows = []
for name, (Zf, yf, weight) in {"no correction": (Z_train, ye_train.to_numpy(), None),
                                "class weights": (Z_train, ye_train.to_numpy(), "balanced"),
                                "SMOTE": (Z_smote, y_smote, None)}.items():
    m = LogisticRegression(class_weight=weight, max_iter=1000).fit(Zf, yf)
    p = m.predict_proba(Z_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(ye_test, (p >= 0.5).astype(int)).ravel()
    rows.append({"method": name, "AUC": roc_auc_score(ye_test, p), "AP": average_precision_score(ye_test, p),
                 "mean predicted risk": p.mean(), "observed rate": ye_test.mean(),
                 "Se": tp / (tp + fn), "PPV": tp / (tp + fp) if tp + fp else np.nan})
pd.DataFrame(rows).round(3)

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-c198)*

**Python note.** `NearestNeighbors(n_neighbors=k + 1).kneighbors(X_min)` returns, for every patient of the rare class, the positions of its nearest patients of the same class. The nearest one is the patient itself, so the function asks for $k + 1$ neighbours and draws among positions 1 to $k$. `rng.integers` draws random whole numbers and `rng.random` random numbers between 0 and 1. `lam * (X_min[j] - X_min[i])` multiplies each row by its own $\lambda$, because NumPy stretches the column `lam` over all variables. `np.vstack` stacks two tables, and `np.concatenate` joins two arrays. The package imbalanced-learn offers the same method as `SMOTE().fit_resample(X, y)`.

**Observe and reflect.** SMOTE and class weights find the same share of events at the threshold 0.5, and their AUC hardly differs from the model without correction. Their mean predicted risk, however, is about eight times the observed rate. What would happen if a clinician read such a probability as the risk of the patient, and what could replace the correction?

**Stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#unsupervised-learning-k-means-and-phenogroups) and continue reading at «Finding subgroups of patients with k-means». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 3: Subgroups, survival and a model to review

*Lecture page: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#stop-3)*

Patients with heart failure differ in more than their ejection fraction. In this stop subgroups are searched for without telling the method the answer, and survival over the follow-up is estimated. Then a model for which an AI assistant reported a perfect result is reviewed.

**At the end of this stop:** The agreement of three clusters with the hidden phenogroups of the clinic (adjusted Rand index 0.706), the event-free survival at 24 months of the patients with HFrEF (0.733) and the honest AUC of the reviewed model (0.849).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 3 · Python basics

*Lecture page: [Finding subgroups of patients with k-means](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#unsupervised-learning-k-means-and-phenogroups)*

The ready application of this stop selects the patients with heart failure, repeats a calculation for a range of values, counts the patients still followed at a given time and multiplies the factors of a Kaplan-Meier curve. The examples below introduce these constructs on a few patients: `isin`, a loop over `range`, counting with a Boolean mask and the Kaplan-Meier estimate by hand.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 3 · Python basics», the cell below repeats the setup of the course package and of the packages of the week, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

In [ ]:
# Python practice: a small example for learning the language
patients = pd.DataFrame({"diagnosis": ["HFrEF", "HFpEF", "CAD", "HFpEF", "AF", "HFrEF"],
                         "lvef": [30, 58, 55, 62, 50, 25]})
hf_only = patients[patients["diagnosis"].isin(["HFrEF", "HFpEF"])]
print(hf_only)
hf_only["diagnosis"].value_counts()

**Python note.** `.isin(["HFrEF", "HFpEF"])` checks every value of a column against a list and gives `True` where the value is in the list. In brackets, this mask keeps the four patients with heart failure. `value_counts()` counts how often each value of a column occurs.

In [ ]:
# Python practice: a small example for learning the language
lvef = np.array([30, 58, 55, 62, 50, 25, 41, 47])
for limit in range(35, 60, 5):   # 35, 40, 45, 50, 55
    print("LVEF <", limit, ":", int(np.sum(lvef < limit)), "patients")

**Python note.** `range(35, 60, 5)` counts from 35 in steps of 5 and stops before 60. `lvef < limit` compares every value of the array with the limit and gives an array of `True` and `False`; `np.sum` counts the `True` values, because Python counts `True` as 1. The ready application chooses the number of clusters with the same kind of loop, `for k in range(2, 7)`.

In [ ]:
# Python practice: a small example for learning the language
time = np.array([2, 3, 3, 5, 8, 12])    # months of follow-up
event = np.array([1, 0, 1, 1, 0, 0])    # 1 = death or hospitalisation, 0 = censored
survival = 1.0
for t in np.unique(time[event == 1]):    # the months with an event, in order
    at_risk = np.sum(time >= t)
    events = np.sum((time == t) & (event == 1))
    survival = survival * (1 - events / at_risk)
    print(f"month {t}: at risk {at_risk}, events {events}, survival {survival:.3f}")

**Python note.** `time[event == 1]` keeps the follow-up times of the patients with an event, and `np.unique` returns them sorted and without repetition: 2, 3 and 5. At each of these months, `np.sum(time >= t)` counts the patients still followed, and `&` joins two masks, so that only an event at month `t` counts; the censored patient of month 3 stays at risk until then but adds no event. The survival is multiplied by the factor of each step: 5/6 = 0.833, then 0.833 × 4/5 = 0.667, then 0.667 × 2/3 = 0.444.

In [ ]:
# Python practice: a small example for learning the language
factors = np.array([1 - 1 / 6, 1 - 1 / 5, 1 - 1 / 3])   # the three steps above
print(np.cumprod(factors).round(3))

**Python note.** `np.cumprod` multiplies the values of an array one after the other and keeps every intermediate product: 0.833, 0.667 and 0.444, the same curve as the loop above, in one line.

**Exercise 3.1.** What is the mean LVEF of the patients with HFrEF or HFpEF in the table `patients` above? Round to one decimal.

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(hf_only["lvef"].mean(), 1)
teach.check("Exercise 3.1", answer)

**Exercise 3.2.** Five patients are followed for 1, 4, 4, 6 and 9 months, and their events are 1, 1, 0, 1 and 0. What is the Kaplan-Meier estimate of survival after month 6? Change `time` and `event` in a copy of the loop above. Round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
time5 = np.array([1, 4, 4, 6, 9])
event5 = np.array([1, 1, 0, 1, 0])
survival5 = 1.0
for t in np.unique(time5[event5 == 1]):
    survival5 *= 1 - np.sum((time5 == t) & (event5 == 1)) / np.sum(time5 >= t)
answer = round(survival5, 3)
teach.check("Exercise 3.2", answer)

**«Stop 3 · Python basics» is complete.** Next comes «Stop 3 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#the-paper-of-the-week) and continue reading at «The paper of the week».

### Stop 3 · Prompt and application

This part has three prompts: «Three clusters», «Kaplan-Meier at 24 months» and «Reviewing the model of an assistant». The repair prompt and the prompt for an error message are in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md) and in the prompts of Stop 1.

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Start cell.** For «Stop 3 · Prompt and application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)
hf = registry[registry["main_diagnosis"].isin(["HFrEF", "HFpEF"])].copy()

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, mean_absolute_error,
                             precision_recall_curve, roc_auc_score, roc_curve,
                             silhouette_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.preprocessing import StandardScaler

#### Prompt 3.1: Three clusters

**Goal.** Let k-means find three groups among the patients with heart failure, then compare them with the hidden phenogroups.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Among the 407 patients with heart failure of the virtual clinic, the cardiologist wants to find subgroups without telling the method any answer, and then to see how well they match the phenogroups with which the clinic was built.  
> Data: The notebook holds a table named "data". Each row is a patient with heart failure: "age" (years), "BMI" (body mass index, kg/m²), "eGFR" (estimated glomerular filtration rate, mL/min/1.73 m²), "LVEF" (left ventricular ejection fraction, %), "NT-proBNP" (pg/mL), "LAVI" (left atrial volume index, mL/m²), "E/e'" (a ratio), "LV mass index" (left ventricular mass index, g/m²) and "TAPSE" (tricuspid annular plane systolic excursion, mm). The column "phenogroup" holds the hidden group with which the clinic generated the patient: "HFrEF", "HFpEF-metabolic" or "HFpEF-cardiorenal".  
> Task: Use the natural logarithm of NT-proBNP in place of NT-proBNP and standardise the nine variables; do not use "phenogroup" for the clustering. Divide the patients into three clusters with k-means. Run the method ten times from different random starting points and keep the best solution; use the number 2026 as the starting value of the random choices. Then compare the clusters with the phenogroups by the adjusted Rand index.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of patients in each cluster and the adjusted Rand index with three decimals, and keep the index under the name "answer".  
> Test: The three clusters must hold the 407 patients together. An index of 1 means perfect agreement, and 0 agreement by chance.  

In [ ]:
# Data of Prompt 3.1: the table "data" that the prompt describes
data = hf[["age", "bmi", "egfr", "lvef", "ntprobnp", "lavi", "e_e_prime", "lv_mass_index", "tapse_mm"]].rename(columns={
    "age": "age", "bmi": "BMI", "egfr": "eGFR", "lvef": "LVEF", "ntprobnp": "NT-proBNP", "lavi": "LAVI",
    "e_e_prime": "E/e'", "lv_mass_index": "LV mass index", "tapse_mm": "TAPSE"})
data["phenogroup"] = hf["hf_phenogroup"].replace({"HFpEF-metabolic": "HFpEF-metabolic", "HFpEF-cardiorenal": "HFpEF-cardiorenal"})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.1
teach.check_step("Prompt 3.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.1 (run it only if the code of the assistant does not pass the check)
data = hf[["age", "bmi", "egfr", "lvef", "ntprobnp", "lavi", "e_e_prime", "lv_mass_index", "tapse_mm"]].rename(columns={
    "age": "age", "bmi": "BMI", "egfr": "eGFR", "lvef": "LVEF", "ntprobnp": "NT-proBNP", "lavi": "LAVI",
    "e_e_prime": "E/e'", "lv_mass_index": "LV mass index", "tapse_mm": "TAPSE"})
data["phenogroup"] = hf["hf_phenogroup"].replace({"HFpEF-metabolic": "HFpEF-metabolic", "HFpEF-cardiorenal": "HFpEF-cardiorenal"})
variables = data.drop(columns="phenogroup").assign(**{"NT-proBNP": np.log(data["NT-proBNP"])})
standardised = StandardScaler().fit_transform(variables)
clusters = KMeans(n_clusters=3, n_init=10, random_state=2026).fit_predict(standardised)
print("Patients per cluster:", np.bincount(clusters))
answer = round(adjusted_rand_score(data["phenogroup"], clusters), 3)
print("Adjusted Rand index:", answer)
teach.check_step("Prompt 3.1", "answer", globals())

**What the result shows.** The clusters recover much of the hidden structure, but not all of it. In real data nobody knows the true groups: Clusters are hypotheses that need clinical sense and outcome data before they mean anything.

#### Prompt 3.2: Kaplan-Meier at 24 months

**Goal.** Estimate which share of the patients with HFrEF are alive and free of a hospitalisation for heart failure two years after the start of follow-up.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to estimate which share of the patients with HFrEF of the virtual clinic are alive and free of a hospitalisation for heart failure 24 months after the start of follow-up.  
> Data: The notebook holds a table named "data". Each row is a patient with heart failure: The column "main diagnosis" says "HFrEF" or "HFpEF". "follow-up" gives the time from the start of follow-up to the first event or to the end of follow-up in months. "event" is 1 for death or a hospitalisation for heart failure, and 0 for a patient who left follow-up without an event (censored).  
> Task: For the patients whose main diagnosis is HFrEF, compute the Kaplan-Meier estimate of event-free survival step by step. Start at 1. At every time at which at least one event occurs, multiply the estimate by 1 − d/n, where d is the number of events at that time and n is the number of patients still followed at that time, those whose follow-up is at least that long. The estimate at 24 months is the value after the last event time of 24 months or less.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the estimate at 24 months with three decimals and keep it under the name "answer".  
> Test: The same calculation for the patients with HFpEF must give a higher value at 24 months.  

In [ ]:
# Data of Prompt 3.2: the table "data" that the prompt describes
data = hf[["main_diagnosis", "followup_months", "event"]].rename(columns={
    "main_diagnosis": "main diagnosis", "followup_months": "follow-up", "event": "event"})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.2
teach.check_step("Prompt 3.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.2 (run it only if the code of the assistant does not pass the check)
data = hf[["main_diagnosis", "followup_months", "event"]].rename(columns={
    "main_diagnosis": "main diagnosis", "followup_months": "follow-up", "event": "event"})


def km_at(time, event, end):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    s = 1.0
    for t in np.unique(time[event == 1]):
        if t > end:
            break
        s *= 1 - np.sum((time == t) & (event == 1)) / np.sum(time >= t)
    return s


hfref = data[data["main diagnosis"] == "HFrEF"]
hfpef = data[data["main diagnosis"] == "HFpEF"]
answer = round(km_at(hfref["follow-up"], hfref["event"], 24), 3)
print("S(24), HFrEF:", answer, " HFpEF:", round(km_at(hfpef["follow-up"], hfpef["event"], 24), 3))
teach.check_step("Prompt 3.2", "answer", globals())

**What the result shows.** In this synthetic registry, about three in four patients with HFrEF are alive and free of a hospitalisation for heart failure after two years. Censored patients count as at risk until they leave follow-up; dropping them would bias the estimate.

#### Prompt 3.3: Reviewing the model of an assistant

**Goal.** An assistant reported an AUC of 1.000 for the event within 12 months. Its model used the length of follow-up and was measured on the patients on which it was trained. Find out what an honest evaluation of the same kind of model gives.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: For the 2000 patients of the virtual clinic, the cardiologist wants an honest estimate of how well a random forest predicts death or a hospitalisation for heart failure within 12 months, after an assistant reported a perfect result.  
> Data: The notebook holds a table named "data". Each row is a patient of the virtual clinic: "age" (years), "eGFR" (estimated glomerular filtration rate, mL/min/1.73 m²), "LVEF" (left ventricular ejection fraction, %), "NT-proBNP" (pg/mL), "AF" (atrial fibrillation, 1 for yes), "diabetes" (1 for yes), "BMI" (body mass index, kg/m²) and "systolic blood pressure" (mm Hg). "follow-up" is the length of follow-up in months, which is known only after the follow-up. "event" is 1 for death or a hospitalisation for heart failure within the first 12 months, and 0 otherwise. "fold" gives the fold of the patient, 1 to 5; the share of events is about the same in every fold.  
> Task: Leave "follow-up" out of the inputs, because it is not known at the time of the decision. Predict "event" from the other eight variables with a random forest. Make the random choices the same in every run by using the number 2026 as the starting value. Measure the model with a five-fold cross-validation: Use each fold once as the test set and the other four folds as the training set, and take the mean AUC of the five folds.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the mean AUC with three decimals and keep it under the name "answer".  
> Test: The cross-validated AUC must be clearly below 1.000. Also print the AUC of the same forest when it is trained and measured on all patients; this value must be clearly higher.  

In [ ]:
# Data of Prompt 3.3: the table "data" that the prompt describes
_r = registry[["age", "egfr", "lvef", "ntprobnp", "af", "diabetes", "bmi", "sbp", "followup_months", "event_12m"]]
data = _r.rename(columns={"age": "age", "egfr": "eGFR", "lvef": "LVEF", "ntprobnp": "NT-proBNP", "af": "AF",
                          "diabetes": "diabetes", "bmi": "BMI", "sbp": "systolic blood pressure",
                          "followup_months": "follow-up", "event_12m": "event"})
data["fold"] = 0
for _k, (_, _rows) in enumerate(StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE).split(_r, _r["event_12m"]), start=1):
    data.iloc[_rows, -1] = _k
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.3
teach.check_range("Prompt 3.3", "answer", 0.65, 0.95, globals())

In [ ]:
#@title Ready code of Prompt 3.3 (run it only if the code of the assistant does not pass the check)
_r = registry[["age", "egfr", "lvef", "ntprobnp", "af", "diabetes", "bmi", "sbp", "followup_months", "event_12m"]]
data = _r.rename(columns={"age": "age", "egfr": "eGFR", "lvef": "LVEF", "ntprobnp": "NT-proBNP", "af": "AF",
                          "diabetes": "diabetes", "bmi": "BMI", "sbp": "systolic blood pressure",
                          "followup_months": "follow-up", "event_12m": "event"})
data["fold"] = 0
for _k, (_, _rows) in enumerate(StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE).split(_r, _r["event_12m"]), start=1):
    data.iloc[_rows, -1] = _k
inputs = data.drop(columns=["follow-up", "event", "fold"])
values = []
for fold in range(1, 6):
    test_rows = data["fold"] == fold
    forest = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=2026)
    forest.fit(inputs[~test_rows], data["event"][~test_rows])
    probabilities = forest.predict_proba(inputs[test_rows])[:, 1]
    values.append(roc_auc_score(data["event"][test_rows], probabilities))
answer = round(float(np.mean(values)), 3)
print("Cross-validated AUC without the follow-up:", answer)
teach.check_range("Prompt 3.3", "answer", 0.65, 0.95, globals())

**What the result shows.** A realistic result for a rare outcome. A perfect AUC is a warning sign: information from the future, such as the length of follow-up, or a test on the patients the model was trained on.

**«Stop 3 · Prompt and application» is complete.** Next comes «Stop 3 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#the-paper-of-the-week) and continue reading at «The paper of the week».

### Stop 3 · Ready application

This part looks for subgroups and follows them over time: k-means on nine standardised variables of the patients with heart failure, with the silhouette, a PCA plot and the comparison with the hidden phenogroups, t-SNE and UMAP maps, the Kaplan-Meier curve and the log-rank test written in NumPy, and the review of a model written by an AI assistant.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 3 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week02 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, mean_absolute_error,
                             precision_recall_curve, roc_auc_score, roc_curve,
                             silhouette_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.preprocessing import StandardScaler

# The three models on the same test set and the same five folds
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

# The registry with follow-up: 2000 patients
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))

# The rare outcome: an event within 12 months
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)

#### Finding subgroups: k-means on the patients with heart failure

*Lecture page: [Finding subgroups of patients with k-means](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#unsupervised-learning-k-means-and-phenogroups)*

The registry holds about 400 patients with heart failure with reduced ejection fraction (HFrEF) or preserved ejection fraction (HFpEF). Unsupervised learning looks for groups without any outcome. The *k-means* algorithm places $k$ centres, assigns each patient to the nearest one, moves each centre to the mean of its patients and repeats until nothing changes. The course hid a phenogroup in every patient with heart failure. The clustering does not see it, and the end of the section compares the clusters with it.

**From the formulas to the code.** k-means minimises the within-cluster sum of squares $WCSS = \sum_{k}\sum_{i \in C_k} \lVert \mathbf{x}_i - \mu_k \rVert^2$, which `KMeans` reports as `inertia_`. The silhouette of a patient is $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$, with $a(i)$ the mean distance to its own cluster and $b(i)$ to the nearest other cluster. `silhouette_score` returns the mean over all patients.

**On the lecture page:** [Worked example: k-means by hand on six ejection fractions](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#worked-example-k-means-by-hand-on-six-ejection-fractions). The cell below makes the same calculation.

In [ ]:
# The worked example of the lecture page: k-means on six ejection fractions
import numpy as np

ef = np.array([25, 30, 35, 55, 60, 65], dtype=float)
centres = np.array([30.0, 35.0])                  # the two starting centres
for step in range(1, 4):
    label = np.argmin(np.abs(ef[:, None] - centres[None, :]), axis=1)   # the nearest centre of each patient
    centres = np.array([ef[label == j].mean() for j in range(2)])
    print("step", step, "| clusters:", label, "| centres:", centres.round(2))
wcss = sum(((ef[label == j] - centres[j]) ** 2).sum() for j in range(2))
a = np.abs(35 - ef[(label == label[2]) & (ef != 35)]).mean()   # mean distance to its own cluster
b = np.abs(35 - ef[label != label[2]]).mean()                 # mean distance to the other cluster
print("WCSS:", wcss, "| silhouette of the patient at 35:", round((b - a) / max(a, b), 2))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-w-km6)*

In [ ]:
# Standardised features of the patients with heart failure
hf = registry[registry["main_diagnosis"].isin(["HFrEF", "HFpEF"])].copy()
HF_FEATURES = ["age", "bmi", "egfr", "lvef", "ntprobnp", "lavi", "e_e_prime", "lv_mass_index", "tapse_mm"]
X_hf = StandardScaler().fit_transform(hf[HF_FEATURES].assign(ntprobnp=np.log(hf["ntprobnp"])))
print(len(hf), "patients with heart failure")

ks = range(2, 7)
wcss, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_hf)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_hf, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].plot(list(ks), wcss, "o-", color="#1F5F8B")
ax[0].set_xlabel("k")
ax[0].set_ylabel("WCSS")
ax[1].plot(list(ks), sil, "o-", color="#9B1B1B")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Mean silhouette")
plt.show()
print(pd.DataFrame({"k": list(ks), "WCSS": np.round(wcss, 1), "silhouette": np.round(sil, 3)}).to_string(index=False))

**Python note.** `.isin([...])` keeps the rows whose value is one of the listed values. `range(2, 7)` gives the numbers 2 to 6, and the loop fits one k-means model for each. `n_init=10` starts each model from ten random positions of the centres and keeps the best result, because k-means can get stuck in a poor solution.

In [ ]:
# Three clusters, drawn on the first two principal components
km3 = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X_hf)
hf["cluster"] = km3.labels_
pca = PCA(n_components=2).fit(X_hf)
pcs = pca.transform(X_hf)
fig, ax = plt.subplots(figsize=(5.6, 4.6))
for c, color in zip(range(3), ["#9B1B1B", "#1F5F8B", "#B97813"]):
    ax.scatter(pcs[hf["cluster"] == c, 0], pcs[hf["cluster"] == c, 1], s=14, color=color, label=f"cluster {c}")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.0f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.0f}%)")
ax.legend()
plt.show()
hf.groupby("cluster")[HF_FEATURES + ["diabetes", "af"]].mean().round(1).T

**Python note.** Principal component analysis (PCA) finds the directions along which the patients differ most. `explained_variance_ratio_` gives the share of the total variance that each direction carries. The plot only shows the clusters, it does not make them: k-means worked on all nine variables.

**Exercise H.** What is the mean silhouette of the k-means solution with three clusters on `X_hf`? Use `silhouette_score(X_hf, km3.labels_)` and round to three decimals.

In [ ]:
data = exercise_table("H", X_hf=X_hf)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(silhouette_score(X_hf, km3.labels_), 3)
teach.check("Exercise H", answer)

In [ ]:
# The hidden truth of the clinic: the phenogroups
print(pd.crosstab(hf["cluster"], hf["hf_phenogroup"]))
print("Adjusted Rand index:", round(adjusted_rand_score(hf["hf_phenogroup"], hf["cluster"]), 3))

**Observe and reflect.** The mean silhouette is slightly higher for two clusters than for three, yet three clusters match the hidden phenogroups well. Which criterion should decide the number of phenogroups in a real study, where nothing is hidden to compare with? Name the clusters by their profile in the table.

**UMAP / t-SNE visualisation.** *Lecture page: [UMAP / t-SNE visualisation](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#umap-and-t-sne-visualisation)*

PCA drew the clusters along the two directions of largest variance. t-SNE and UMAP draw a map that keeps the neighbourhoods of the patients instead. The cell draws three t-SNE maps with perplexities of 5, 30 and 50, coloured by the k-means clusters, and measures how well the clusters stay apart on each map with the silhouette. The clusters were found in the nine variables, not on the maps.

In [ ]:
# t-SNE maps of the patients with heart failure at three perplexities
COLORS = ["#9B1B1B", "#1F5F8B", "#B97813"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, perp in zip(axes, [5, 30, 50]):
    emb = TSNE(n_components=2, perplexity=perp, init="pca", random_state=RANDOM_STATE).fit_transform(X_hf)
    for c, color in zip(range(3), COLORS):
        ax.scatter(emb[hf["cluster"] == c, 0], emb[hf["cluster"] == c, 1], s=10, color=color, label=f"cluster {c}")
    ax.set_title(f"t-SNE, perplexity {perp}")
    ax.set_xticks([])
    ax.set_yticks([])
    print(f"perplexity {perp}: silhouette of the clusters on the map {silhouette_score(emb, hf['cluster']):.3f}")
axes[0].legend()
plt.show()

**Python note.** `TSNE(n_components=2, perplexity=...)` makes a two-dimensional map, and `fit_transform` returns one row of two coordinates per patient. `init="pca"` starts the map from the PCA picture, which makes it more stable, and `random_state` fixes the random start. The axes of a t-SNE map have no unit, so the cell removes their ticks.

In [ ]:
# UMAP, when the umap-learn package is installed
import warnings

try:
    import umap
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        emb_u = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=RANDOM_STATE).fit_transform(X_hf)
    fig, ax = plt.subplots(figsize=(5, 4.2))
    for c, color in zip(range(3), COLORS):
        ax.scatter(emb_u[hf["cluster"] == c, 0], emb_u[hf["cluster"] == c, 1], s=10, color=color, label=f"cluster {c}")
    ax.set_title("UMAP, 15 neighbours")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend()
    plt.show()
    print("Silhouette of the clusters on the map:", round(silhouette_score(emb_u, hf["cluster"]), 3))
except ImportError:
    print("umap-learn is not installed; pip install umap-learn installs it. The t-SNE maps above show the same idea.")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-c217)*

**Python note.** `umap.UMAP(n_neighbors=15, min_dist=0.1)` builds a graph of the 15 nearest neighbours of each patient and lays it out in two dimensions; `min_dist` sets how tightly the points may pack. The first run takes about half a minute, because the package compiles its code. `warnings.catch_warnings()` hides the technical warnings of the package inside the `with` block.

**Observe and reflect.** On the t-SNE maps, the clusters look more separate than on the PCA plot, and their shapes change with the perplexity. Which of the two pictures shows the distances between the clusters more faithfully? Why would it be wrong to run k-means on the t-SNE coordinates?

#### Survival analysis: Kaplan-Meier and the log-rank test

*Lecture page: [Survival analysis: The Kaplan-Meier curve](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#survival-analysis-kaplan-meier-and-the-log-rank-test)*

Follow-up ends for different patients at different times. A patient still free of events at the last visit is *censored*: The event may come later, but it was not seen. The Kaplan-Meier estimator uses every patient for as long as the patient was followed. The log-rank test asks whether two or more curves differ more than chance would explain.

**From the formulas to the code.** At each time $t_i$ with an event, $n_i$ patients are at risk and $d_i$ have the event. The estimate is $S(t) = \prod_{t_i \le t} \left(1 - \frac{d_i}{n_i}\right)$. The log-rank statistic is $X^2 = \sum_g \frac{(O_g - E_g)^2}{E_g}$, with $O_g$ the observed events of group $g$ and $E_g = \sum_i d_i\, n_{gi}/n_i$ the expected ones. With $G$ groups it is compared with a chi-square distribution with $G - 1$ degrees of freedom.

**Start cell.** For the section «Survival analysis: Kaplan-Meier and the log-rank test», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Survival analysis: Kaplan-Meier and the log-rank test»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week02 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# The registry with follow-up: 2000 patients
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))

# The rare outcome: an event within 12 months
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)

# Standardised features of the patients with heart failure
hf = registry[registry["main_diagnosis"].isin(["HFrEF", "HFpEF"])].copy()
HF_FEATURES = ["age", "bmi", "egfr", "lvef", "ntprobnp", "lavi", "e_e_prime", "lv_mass_index", "tapse_mm"]
X_hf = StandardScaler().fit_transform(hf[HF_FEATURES].assign(ntprobnp=np.log(hf["ntprobnp"])))

# Three clusters, drawn on the first two principal components
km3 = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X_hf)
hf["cluster"] = km3.labels_

In [ ]:
# Kaplan-Meier by hand
def kaplan_meier(time, event):
    """Times of the steps and survival after each step, starting at (0, 1)."""
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    times, surv, s = [0.0], [1.0], 1.0
    for t in np.unique(time[event == 1]):
        at_risk = np.sum(time >= t)
        events = np.sum((time == t) & (event == 1))
        s = s * (1 - events / at_risk)
        times.append(t)
        surv.append(s)
    return np.array(times), np.array(surv)


def survival_at(times, surv, t):
    """S(t) read from the step function."""
    return surv[np.searchsorted(times, t, side="right") - 1]


t10 = [2, 3, 3, 5, 6, 8, 9, 11, 12, 12]
e10 = [1, 1, 0, 1, 0, 1, 0, 1, 0, 0]
tt, ss = kaplan_meier(t10, e10)
pd.DataFrame({"month": tt, "S(t)": ss.round(4)})

**Python note.** `np.unique` returns the sorted distinct times with an event. `np.sum(time >= t)` counts the patients still followed at time `t`, which is $n_i$, and the condition `(time == t) & (event == 1)` counts the events at that time, $d_i$. `np.searchsorted` finds where a time falls among the steps of the curve.

**On the lecture page:** [Worked example: Ten patients by hand](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#worked-example-ten-patients-by-hand). The cell below gives the same ten patients to the function `kaplan_meier` above.

In [ ]:
# The worked example of the lecture page: ten patients by hand
time10 = np.array([2, 3, 3, 5, 6, 8, 9, 11, 12, 12])   # months of follow-up
event10 = np.array([1, 1, 0, 1, 0, 1, 0, 1, 0, 0])     # 1 = event, 0 = censored
tt, ss = kaplan_meier(time10, event10)
pd.DataFrame({"month": tt, "survival": np.round(ss, 3)})

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#nbref-w-km10)*

In [ ]:
# Kaplan-Meier curves by cluster and by the type of heart failure
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.9), sharey=True)
for c, color in zip(range(3), ["#9B1B1B", "#1F5F8B", "#B97813"]):
    d = hf[hf["cluster"] == c]
    tt, ss = kaplan_meier(d["followup_months"], d["event"])
    ax[0].step(tt, ss, where="post", color=color, label=f"cluster {c} (n={len(d)})")
for kind, color in zip(["HFrEF", "HFpEF"], ["#9B1B1B", "#1F5F8B"]):
    d = hf[hf["main_diagnosis"] == kind]
    tt, ss = kaplan_meier(d["followup_months"], d["event"])
    ax[1].step(tt, ss, where="post", color=color, label=f"{kind} (n={len(d)})")
    print(kind, "S(24) =", round(survival_at(tt, ss, 24), 3))
for a in ax:
    a.set_xlabel("Months")
    a.legend(loc="lower left")
ax[0].set_ylabel("Free of death or hospitalisation")
plt.show()


def logrank(time, event, group):
    """Log-rank test for two or more groups: chi-square and p value."""
    time, event, group = np.asarray(time, float), np.asarray(event, int), np.asarray(group)
    labels = np.unique(group)
    observed = np.zeros(len(labels))
    expected = np.zeros(len(labels))
    for t in np.unique(time[event == 1]):
        at_risk = time >= t
        d = np.sum((time == t) & (event == 1))
        n = np.sum(at_risk)
        for j, g in enumerate(labels):
            observed[j] += np.sum((time == t) & (event == 1) & (group == g))
            expected[j] += d * np.sum(at_risk & (group == g)) / n
    chi2 = np.sum((observed - expected) ** 2 / expected)
    return chi2, stats.chi2.sf(chi2, len(labels) - 1)


for name, col in [("HFrEF vs HFpEF", "main_diagnosis"), ("clusters", "cluster")]:
    chi2, p = logrank(hf["followup_months"], hf["event"], hf[col])
    print(f"{name}: X2 = {chi2:.2f}, p = {p:.2g}")

**Python note.** `ax[0].step(..., where="post")` draws a step function that keeps each value until the next time, the way a Kaplan-Meier curve looks. `stats.chi2.sf(x, df)` from SciPy gives the probability of a chi-square value at least as large as `x`, which is the p value. `f"{p:.2g}"` prints a number with two significant digits.

**Exercise I.** What is the Kaplan-Meier estimate of being free of death or hospitalisation at 24 months for the patients with HFrEF? Round to three decimals.

In [ ]:
data = exercise_table("I", hf=hf)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
d = hf[hf["main_diagnosis"] == "HFrEF"]
tt, ss = kaplan_meier(d["followup_months"], d["event"])
answer = round(survival_at(tt, ss, 24), 3)
teach.check("Exercise I", answer)

**Observe and reflect.** HFrEF and HFpEF do not differ much in this registry, while the three clusters do. What does this say about grouping patients by the ejection fraction alone? Which cluster would you follow more closely, and what else would you want to know before changing its care?

**Advanced (optional).** If the package `lifelines` is installed, compare the two functions of this section with `KaplanMeierFitter` and `multivariate_logrank_test` of `lifelines`. The log-rank statistic of `lifelines` uses the exact variance and is therefore slightly larger.

In [ ]:
# Advanced (optional)
try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import multivariate_logrank_test
    d = hf[hf["main_diagnosis"] == "HFrEF"]
    kmf = KaplanMeierFitter().fit(d["followup_months"], d["event"])
    print("lifelines S(24):", round(float(kmf.survival_function_at_times(24).iloc[0]), 3))
    print(multivariate_logrank_test(hf["followup_months"], hf["cluster"], hf["event"]).summary)
except ImportError:
    print("lifelines is not installed; the functions of this section are enough for the course.")

#### Working with an AI assistant, part 2: Reviewing a model

*Lecture page: [Working with an AI assistant, part 2: Reviewing a model](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#ai-pair-programming-part-2-reviewing-a-model-built-by-an-assistant)*

A participant asked an AI assistant for *a model that predicts death or hospitalisation within 12 months from the registry, with its AUC*. The code below is what came back. It runs without an error and reports an excellent result. Read it with the model review questions of the lecture before running it: Which variables may the model see, and on which patients is it measured?

In [ ]:
# Code from an AI assistant: predict the 12-month event and report the AUC
features = ["age", "egfr", "lvef", "ntprobnp", "af", "diabetes", "bmi", "sbp", "followup_months"]
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(registry[features], registry["event_12m"])
auc_reported = roc_auc_score(registry["event_12m"], rf.predict_proba(registry[features])[:, 1])
print("AUC:", round(auc_reported, 3))

**Observe and reflect.** The code has two problems. The first is a variable that the model could never know at the time of the decision. The second is where the AUC is measured. Name both, and say which question of the review catches each.

**Exercise J.** Correct the code: Leave `followup_months` out, use `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)` and report the mean AUC of a five-fold cross-validation with `StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)`. Round to three decimals.

In [ ]:
data = exercise_table("J", cv=cv, registry=registry)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
fixed = [f for f in features if f != "followup_months"]
forest_cv = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
answer = round(cross_val_score(forest_cv, registry[fixed], registry["event_12m"], cv=folds, scoring="roc_auc").mean(), 3)
teach.check("Exercise J", answer)

The [prompt workshop](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lab.html#part-f) of the interactive lab writes the prompt for such a task with the seven fields of the prompt card. The field *Test* is the place for the check that would have caught both problems: a test of the model on patients it has not seen.

**Stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#the-paper-of-the-week) and continue reading at «The paper of the week». The weekly project is at the end of this Jupyter notebook, and the section «Weekly project» of the lecture page describes it.

## Application key: Another outcome, other variables

*Lecture page: [With your own data](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#with-your-own-data)*

Change `OUTCOME` and `VARIABLES` and run the cell again. It reports the cross-validated AUC of a logistic regression and of a random forest, and the odds ratios of the logistic regression. Outcomes worth a try: `"cad"`, `"event_12m"`, `"af"` and `"event"`. For each, think first about which variables would be known at the time of the decision.

This part follows the practice stops and stands on its own: Its start cell repeats the setup and the data calls that its cells use.

**Start cell.** For «Application key: Another outcome, other variables», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Application key: Another outcome, other variables»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# The registry with follow-up: 2000 patients
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))

# The rare outcome: an event within 12 months
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)

In [ ]:
OUTCOME = "af"
VARIABLES = ["age", "bmi", "hypertension", "lvef", "lavi", "ntprobnp"]

Xo = registry[VARIABLES].copy()
if "ntprobnp" in Xo:
    Xo["ntprobnp"] = np.log(Xo["ntprobnp"])
yo = registry[OUTCOME]
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
lr_o = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
rf_o = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)
print("Prevalence of", OUTCOME, round(yo.mean(), 3))
for name, m in [("logistic regression", lr_o), ("random forest", rf_o)]:
    s = cross_val_score(m, Xo, yo, cv=folds, scoring="roc_auc")
    print(f"{name}: AUC {s.mean():.3f} ± {s.std():.3f}")
lr_o.fit(Xo, yo)
pd.Series(np.exp(lr_o.named_steps["logisticregression"].coef_[0]), index=VARIABLES,
          name="odds ratio per SD").round(2)

**«Application key: Another outcome, other variables» is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#with-your-own-data).

## Weekly project

*Lecture page: [Weekly project](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#weekly-project) · [README of Week 2, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/README.md#weekly-project)*

**The project.** Phenogroups of heart failure in the registry of the virtual clinic: The patients with HFrEF or HFpEF are clustered with k-means, the clusters are described and named, their prognosis is compared with Kaplan-Meier curves and the log-rank test, and the clusters are compared with the phenogroups that the course hid in the registry.

This part stands on its own: Its start cell loads the data of the project and prepares the tables of the steps. The project can be built with the prompts below, one for each step; each step works like a prompt of the stops, without a check cell or a ready code, because the results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log.

**Start cell.** For the weekly project, the cell below repeats the setup and the data calls of this part, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the weekly project
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=2, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
_hf = registry[registry["main_diagnosis"].isin(["HFrEF", "HFpEF"])].copy()
_HF_COLS = {"age": "age", "bmi": "BMI", "egfr": "eGFR", "lvef": "LVEF", "ntprobnp": "NT-proBNP", "lavi": "LAVI",
            "e_e_prime": "E/e'", "lv_mass_index": "LV mass index", "tapse_mm": "TAPSE"}


def _project_k():
    """The number of clusters chosen in project step 1, or 3."""
    k = globals().get("answer")
    k = int(k) if isinstance(k, (int, np.integer)) and 2 <= int(k) <= 6 else globals().get("_chosen_k", 3)
    globals()["_chosen_k"] = k
    return k


def _project_clusters(k):
    z = StandardScaler().fit_transform(_hf[list(_HF_COLS)].assign(ntprobnp=np.log(_hf["ntprobnp"])))
    return KMeans(n_clusters=k, n_init=10, random_state=2026).fit_predict(z) + 1

### Project step 1: Choose the number of phenogroups

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Among the 407 patients with heart failure of the virtual clinic, the cardiologist wants to find phenogroups and to choose their number.  
> Data: The notebook holds a table named "data". Each row is a patient with heart failure: "age" (years), "BMI" (body mass index, kg/m²), "eGFR" (estimated glomerular filtration rate, mL/min/1.73 m²), "LVEF" (left ventricular ejection fraction, %), "NT-proBNP" (pg/mL), "LAVI" (left atrial volume index, mL/m²), "E/e'" (a ratio), "LV mass index" (left ventricular mass index, g/m²) and "TAPSE" (tricuspid annular plane systolic excursion, mm).  
> Task: Use the natural logarithm of NT-proBNP and standardise the nine variables. For each number of clusters from 2 to 6, divide the patients with k-means; run the method ten times from different random starting points, keep the best solution and use the number 2026 as the starting value. For each number of clusters, compute the within-cluster sum of squares (WCSS) and the mean silhouette. Choose the number of clusters and give the clinical reason for the choice in one or two sentences.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a table of the number of clusters, the WCSS and the silhouette, and keep the chosen number of clusters, a whole number, under the name "answer".  
> Test: The WCSS must fall as the number of clusters rises, and every silhouette must lie between −1 and 1.  

In [ ]:
# Data of project step 1: the table "data" that the prompt describes
data = _hf[list(_HF_COLS)].rename(columns=_HF_COLS)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: Describe and name the phenogroups

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to describe the phenogroups of the patients with heart failure and to give each a short clinical name.  
> Data: The notebook holds a table named "data". Each row is a patient with heart failure: "cluster" gives the phenogroup found with the number of clusters chosen in project step 1. The other columns hold "age" (years), "BMI" (kg/m²), "eGFR" (mL/min/1.73 m²), "LVEF" (%), "NT-proBNP" (pg/mL), "LAVI" (mL/m²), "E/e'", "LV mass index" (g/m²), "TAPSE" (mm), "female" (1 for women), "diabetes" and "AF" (1 for yes) and "main diagnosis" ("HFrEF" or "HFpEF").  
> Task: Build a table with one column per cluster: the number of patients; the mean of age, BMI, eGFR, LVEF, LAVI, E/e', LV mass index and TAPSE; the median of NT-proBNP; and the percentage of women, of diabetes, of AF and of HFrEF. Then propose a short clinical name for each cluster.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the table and the names, and keep the table under the name "answer".  
> Test: The numbers of patients in the table must add up to 407.  

In [ ]:
# Data of project step 2: the table "data" that the prompt describes
data = _hf[list(_HF_COLS)].rename(columns=_HF_COLS)
data.insert(0, "cluster", _project_clusters(_project_k()))
data["female"] = (_hf["sex"] == "F").astype(int)
data["diabetes"] = _hf["diabetes"]
data["AF"] = _hf["af"]
data["main diagnosis"] = _hf["main_diagnosis"]
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: Prognosis by phenogroup

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to compare the prognosis of the phenogroups with that of the two classical groups, HFrEF and HFpEF.  
> Data: The notebook holds a table named "data". Each row is a patient with heart failure: "cluster" gives the phenogroup of project step 1, and "main diagnosis" says "HFrEF" or "HFpEF". "follow-up" gives the time to the first event or to the end of follow-up in months, and "event" is 1 for death or a hospitalisation for heart failure and 0 for a patient who left follow-up without an event (censored).  
> Task: Draw the Kaplan-Meier curves of event-free survival of the clusters in one panel, and of HFrEF and HFpEF in a second panel. Compute the survival at 24 months of every group, and compare the groups of each panel with a log-rank test.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the figure, the table of the survival at 24 months and the p values of the two log-rank tests, and keep the p value of the clusters under the name "answer".  
> Test: Every curve must start at 1 and never rise.  

In [ ]:
# Data of project step 3: the table "data" that the prompt describes
data = pd.DataFrame({"cluster": _project_clusters(_project_k()), "main diagnosis": _hf["main_diagnosis"].to_numpy(),
                     "follow-up": _hf["followup_months"].to_numpy(), "event": _hf["event"].to_numpy()})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: Compare with the hidden truth

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The virtual clinic hid a phenogroup in every patient with heart failure. The cardiologist wants to know how well the clusters found in project step 1 recover them.  
> Data: The notebook holds a table named "data". Each row is a patient with heart failure: "cluster" gives the cluster of project step 1, and "phenogroup" the hidden phenogroup, "HFrEF", "HFpEF-metabolic" or "HFpEF-cardiorenal".  
> Task: Cross-tabulate the clusters with the phenogroups and compute the adjusted Rand index. Then write three sentences on what the clustering found that the ejection fraction alone misses.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the cross-table, the index with three decimals and the three sentences, and keep the index under the name "answer".  
> Test: The index must lie between −1 and 1; with three clusters it should be clearly above 0.5.  

In [ ]:
# Data of project step 4: the table "data" that the prompt describes
data = pd.DataFrame({"cluster": _project_clusters(_project_k()),
                     "phenogroup": _hf["hf_phenogroup"].replace({"HFpEF-metabolic": "HFpEF-metabolic", "HFpEF-cardiorenal": "HFpEF-cardiorenal"}).to_numpy()})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

**End of the weekly project.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#weekly-project): The section «Weekly project» describes the report and its submission.

## Application challenges by field

| Field | Challenge |
|---|---|
| Heart failure | Add the follow-up NT-proBNP of your own clinic's protocol as a thought experiment: Which of the nine clustering variables would you replace to make the phenogroups easier to use at the bedside? Cluster again with your choice and compare the Kaplan-Meier curves. |
| Interventional cardiology | Build a model of coronary artery disease from the registry with the variables of the risk outpatient clinic of «Stop 2 · Ready application», choose the threshold at which it would send no more than 20 percent of the patients without disease to angiography, and report its sensitivity at that threshold. |
| Electrophysiology | Predict atrial fibrillation (`af`) from age, BMI, hypertension, left atrial volume index and NT-proBNP. Which variable carries most of the information, and does the random forest add anything to the logistic regression? |
| Cardiovascular imaging | Cluster the patients with heart failure with the echocardiographic variables only (`lvef`, `lavi`, `e_e_prime`, `lv_mass_index`, `tapse_mm`) and compare the agreement with the hidden phenogroups with that of the nine variables. |
| Preventive cardiology | Draw the predicted probability of coronary artery disease of `cad_model` against age for a smoker and a non-smoker with otherwise average values. How many years of age does smoking add in this model? |
| Cardio-oncology | Imagine a screening model for cardiotoxicity with a sensitivity of 0.90 and a specificity of 0.80. Plot its PPV for prevalences from 1 to 20 percent and decide at which prevalence it would be worth using. |
| Intensive care | Use the 12-month event model with class weights and choose the threshold that finds 80 percent of the events. How many alarms does it raise per 100 patients, and how many of them are real events? |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of Week 2, described on the [week page](README.md), starts from the clusters and the two functions of «Stop 3 · Ready application».

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>